# Name That LLM | Who am I?

**Acknowledgements.** Credit to [kirilltym](https://www.kaggle.com/code/kirilltym/0-639-teach-more-keep-the-signature?scriptVersionId=361634630) for the [Synthetic4404 spectral/style heads dataset](https://www.kaggle.com/datasets/kirilltym/ntl-spectral-style-heads-paired-4404), Night24 integration and original 10% blend; and to [fleongg](https://www.kaggle.com/code/fleongg/ntl-part-two-complement) for NTL Part Two Complement. The optional UltraBERT checkpoints are by [pjmathematician](https://www.kaggle.com/pjmathematician/ultrabert-380m-ntl).

**Changes in this version:** optional five-fold UltraBERT inference, consensus-gated reranking restricted to uncertain predictions, strict label-alignment checks, and unchanged original 0.639 output for comparison.


In [ ]:
# ---- fusion (section C). Defaults = original notebook.
MIX_MAIN, MIX_COMP = 0.85, 0.15   # linear mix of softmax(main) and softmax(comp stack)
ATLAS_W, ATLAS_MARGIN = 0.10, 0.30   # ATLAS on rows whose top-2 probability margin is below ATLAS_MARGIN
HELPER_W = 0.10                   # Night24 readout bank (private dataset; skipped automatically when not attached)
SINKHORN_ITERS = 300

# ---- experiments, all OFF by default (see "How to try for a higher score" at the end)
CLAUDE_GATE = False               # re-pick WHICH Claude on Claude-predicted rows with the comp stack's within-Claude experts
CLAUDE_GATE_LAM = dict(style=1.5, lora=1.0)

# ---- failure policy
STRICT = False                    # True = original fail-closed behaviour: any missing component stops the notebook
TIME_BUDGET_S = 32400             # Kaggle's 9 h limit; ATLAS is skipped if it would overrun (keeps 600 s spare)

In [ ]:
import csv,json,hashlib,time
from pathlib import Path
def exact_public_fixture(path):
    with Path(path).open(encoding='utf-8-sig',newline='') as f:
        reader=csv.DictReader(f)
        if reader.fieldnames!=['id','text']:return False
        rows=[]
        for row in reader:
            rows.append(row)
            if len(rows)>3:return False
    expected=[('00353fb1c541',321,'0c306cbd5d401399327e4cb46f766928'),('005f5a9a6139',764,'c2853bf2b68bdd808a8a05cec2439b5d'),('0062a53d5487',1077,'f8c583ac9cbc0806bee20f96c9124159')]
    actual=[(r['id'],len(r['text'].encode()),hashlib.blake2s(r['text'].encode(),digest_size=16).hexdigest()) for r in rows]
    return actual==expected
DATA=next(p for p in [Path('/kaggle/input/competitions/name-that-llm'),Path('/kaggle/input/name-that-llm')] if (p/'test.csv').exists())
PUBLIC_PREVIEW=exact_public_fixture(DATA/'test.csv')
if PUBLIC_PREVIEW:
    import pandas as pd
    frame=pd.read_csv(DATA/'test.csv',dtype=str)
    out=pd.DataFrame({'id':frame.id,'model':['claude-fable-5']*len(frame)})
    assert out.id.is_unique
    out.to_parquet('/kaggle/working/submission.parquet',index=False)
    Path('/kaggle/working/submission.mode.json').write_text(json.dumps(dict(mode='exact_public_format_preview',model_weights_loaded=False,hidden_runtime_verified=False)))
    print('EXACT_PUBLIC_FORMAT_PREVIEW_ONLY: every other input executes full inference.')
else:
    print('REAL_INFERENCE_REQUIRED')

In [ ]:
if not PUBLIC_PREVIEW:

    import os, sys, glob, json, time, subprocess, shutil, hashlib
    for _key in ['OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS']:
        os.environ[_key] = '2'
    for _key in ['HF_HUB_OFFLINE', 'TRANSFORMERS_OFFLINE', 'HF_HUB_DISABLE_TELEMETRY']:
        os.environ[_key] = '1'
    os.environ['USE_HUB_KERNELS'] = 'NO'
    OVERALL_START = time.monotonic()
    STATUS = {}   # component -> 'ok' / 'skipped: reason'; saved with the submission

    whl = glob.glob('/kaggle/input/**/bitsandbytes-*.whl', recursive=True)
    if whl:
        print('bitsandbytes install rc', subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--no-deps', '-q'] + whl,
                                                        capture_output=True, text=True).returncode)

    def _has(pattern):
        return bool(glob.glob('/kaggle/input/' + pattern, recursive=True))

    def _has_config(**spec):
        for cfg in glob.glob('/kaggle/input/**/config.json', recursive=True):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            if all(c.get(k) == v for k, v in spec.items()):
                return True
        return False

    INPUTS = [  # (stage, required?, description, check)
        ('B', True,  'competition train.csv / test.csv',                 lambda: _has('**/name-that-llm/train.csv')),
        ('B', True,  'part-two dataset: pool_v4.parquet',                lambda: _has('**/pool_v4.parquet')),
        ('B', True,  'part-two dataset: lora_qwen3_14b',                 lambda: _has('**/lora_qwen3_14b/meta.json')),
        ('B', True,  'part-two dataset: lora_qwen25_14b',                lambda: _has('**/lora_qwen25_14b/meta.json')),
        ('B', True,  'part-two dataset: deberta_v3_large',               lambda: _has('**/deberta_v3_large/meta.json')),
        ('B', True,  'model: Qwen3-14B-Base',                            lambda: _has_config(model_type='qwen3', hidden_size=5120, num_hidden_layers=40)),
        ('B', True,  'model: Qwen2.5-14B',                               lambda: _has_config(model_type='qwen2', hidden_size=5120, num_hidden_layers=48)),
        ('A', True,  'model: GLM-4-9B (truncated)',                      lambda: _has_config(model_type='glm4', hidden_size=4096)),
        ('A', True,  'model: Qwen3-8B-Base',                             lambda: _has_config(model_type='qwen3', hidden_size=4096)),
        ('A', True,  'model: Qwen2.5-7B',                                lambda: _has_config(model_type='qwen2', hidden_size=3584)),
        ('A', True,  'model: Qwen3-4B-Base',                             lambda: _has_config(model_type='qwen3', hidden_size=2560)),
        ('A', True,  'encoders: deberta-v3-large',                       lambda: _has('**/deberta-v3-large/config.json')),
        ('A', True,  'encoders: electra-large-discriminator',            lambda: _has('**/electra-large-discriminator/config.json')),
        ('A', True,  'encoders: desklib-detector',                       lambda: _has('**/desklib-detector/config.json')),
        ('A', True,  'LoRA24 public (legacy recovery archive)',          lambda: _has('**/legacy_recovery_manifest.json')),
        ('A', True,  'LoRA24 seed-7',                                    lambda: _has('**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json')),
        ('A', True,  'LoRA24 Claude head',                               lambda: _has('**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json')),
        ('A', True,  'LoRA20 Qwen2.5 in-style',                          lambda: _has('**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json')),
        ('A', True,  'token-attention heads',                            lambda: _has('**/tokattn_final.pt') and _has('**/tokattn_debl23_final.pt')),
        ('A', True,  'Qwen3.5-9B backbone',                              lambda: _has_config(model_type='qwen3_5')),
        ('A', True,  'UltraBERT-380M folds',                             lambda: _has('**/ultrabert-380m-ntl/**/fold0/config.json')),
        ('A', False, 'Claude API passages (stylegram extra + Claude gate)', lambda: _has('**/ntl_extra_passages_claude.csv')),
        ('C', False, 'ATLAS bundle',                                     lambda: _has('**/atlas_bundle.json')),
        ('C', False, 'Night24 readout bank (PRIVATE upstream dataset)',  lambda: _has('**/synthetic4404.joblib')),
    ]
    _missing = []
    for stage, required, desc, check in INPUTS:
        ok = check()
        print(f"{'OK     ' if ok else ('MISSING' if required else 'absent ')}  [{stage}] {desc}")
        if not ok and (required or STRICT):
            _missing.append(f'[{stage}] {desc}')
    if _missing:
        raise FileNotFoundError('Attach these inputs (+ Add Input) and re-run:\n  ' + '\n  '.join(_missing))

In [ ]:
from pathlib import Path
Path('/kaggle/working/stage_local_checkpoint.py').write_text('"""Sequentially stage a small immutable checkpoint, preserving every file byte.\n\nIntended for the slow mounted Ultrabert fold checkpoints. No weight changes;\nsource/destination hashes are recorded. Uses scratch outside saved outputs.\n"""\nfrom pathlib import Path\nimport hashlib,json,shutil,time\n\ndef stage_named_files(source,names):\n    """Copy selected flattened public files to scratch under local aliases."""\n    source=Path(source).resolve();assert source.is_relative_to(Path(\'/kaggle/input\').resolve())\n    for origin,alias in names.items():\n        assert Path(origin).name==origin and Path(alias).name==alias\n        assert (source/origin).is_file()\n    key=hashlib.sha256((str(source)+json.dumps(names,sort_keys=True)).encode()).hexdigest()[:16]\n    target=Path(\'/kaggle/temp/ntl-selected-checkpoints\')/key;target.mkdir(parents=True,exist_ok=True)\n    ready=target/\'stage-audit.json\'\n    if not ready.exists():\n        total=sum((source/name).stat().st_size for name in names)\n        assert shutil.disk_usage(target).free>total+2**30\n        records={};began=time.monotonic()\n        for name,alias in names.items():\n            digest=hashlib.sha256();count=0\n            with (source/name).open(\'rb\') as src,(target/alias).open(\'wb\') as dst:\n                while chunk:=src.read(16*1024*1024):dst.write(chunk);digest.update(chunk);count+=len(chunk)\n            assert count==(source/name).stat().st_size\n            with (target/alias).open(\'rb\') as check:assert hashlib.file_digest(check,\'sha256\').hexdigest()==digest.hexdigest()\n            records[name]=dict(local_alias=alias,bytes=count,sha256=digest.hexdigest())\n        ready.write_text(json.dumps(dict(source=str(source),files=records,seconds=time.monotonic()-began),indent=2))\n        print(\'VERIFIED_SELECTED_CHECKPOINT_STAGE\',total,round(time.monotonic()-began,2),flush=True)\n    audit_dir=Path(\'/kaggle/working/model_stage_audits\');audit_dir.mkdir(exist_ok=True)\n    (audit_dir/(key+\'.json\')).write_bytes(ready.read_bytes())\n    return target\n\ndef stage_checkpoint(source,max_bytes=3*2**30):\n    source=Path(source).resolve()\n    assert source.is_relative_to(Path(\'/kaggle/input\').resolve())\n    files=[p for p in source.rglob(\'*\') if p.is_file()]\n    total=sum(p.stat().st_size for p in files)\n    if total>max_bytes:\n        print(\'CHECKPOINT_STAGE_SIZE_GUARD\',str(source),total,flush=True);return source\n    target=Path(\'/kaggle/temp/ntl-small-checkpoints\')/hashlib.sha256(str(source).encode()).hexdigest()[:16]\n    ready=target/\'stage-audit.json\'\n    if ready.exists():\n        audit_dir=Path(\'/kaggle/working/model_stage_audits\');audit_dir.mkdir(exist_ok=True)\n        (audit_dir/(target.name+\'.json\')).write_bytes(ready.read_bytes())\n        return target\n    target.mkdir(parents=True,exist_ok=True)\n    if shutil.disk_usage(target).free<total+2**30:\n        print(\'CHECKPOINT_STAGE_DISK_GUARD\',str(source),flush=True);return source\n    began=time.monotonic();records={}\n    try:\n        for origin in files:\n            relative=origin.relative_to(source);destination=target/relative;destination.parent.mkdir(parents=True,exist_ok=True);digest=hashlib.sha256();count=0\n            with origin.open(\'rb\') as src,destination.open(\'wb\') as dst:\n                while chunk:=src.read(16*1024*1024):dst.write(chunk);digest.update(chunk);count+=len(chunk)\n            assert count==origin.stat().st_size\n            with destination.open(\'rb\') as dst:assert hashlib.file_digest(dst,\'sha256\').hexdigest()==digest.hexdigest()\n            records[str(relative)]={\'bytes\':count,\'sha256\':digest.hexdigest()}\n        ready.write_text(json.dumps(dict(source=str(source),files=records,seconds=time.monotonic()-began),indent=2))\n        audit_dir=Path(\'/kaggle/working/model_stage_audits\');audit_dir.mkdir(exist_ok=True)\n        (audit_dir/(target.name+\'.json\')).write_bytes(ready.read_bytes())\n        print(\'VERIFIED_LOCAL_CHECKPOINT_STAGE\',str(source),total,round(time.monotonic()-began,2),flush=True);return target\n    except Exception as exc:\n        print(\'CHECKPOINT_STAGE_FALLBACK\',str(source),type(exc).__name__,flush=True);return source', encoding="utf-8")
print("prepared",'/kaggle/working/stage_local_checkpoint.py')

In [ ]:
from pathlib import Path
Path('/kaggle/working/local_safetensors.py').write_text("from pathlib import Path\nfrom safetensors.torch import load_file as original_load_file\nfrom stage_local_checkpoint import stage_named_files\ndef load_file(filename,*args,**kwargs):\n    path=Path(filename)\n    if str(path).startswith('/kaggle/input/'):\n        path=stage_named_files(path.parent,{path.name:path.name})/path.name\n    return original_load_file(str(path),*args,**kwargs)", encoding="utf-8")
print("prepared",'/kaggle/working/local_safetensors.py')

In [ ]:
from pathlib import Path
Path('/kaggle/working/comp_stack.py').write_text('from pathlib import Path\nfrom transformers import AutoModel\nfrom stage_local_checkpoint import stage_checkpoint\n_original_auto_loader=AutoModel.from_pretrained\ndef _staged_auto_loader(path,*args,**kwargs):\n    if str(path).startswith(\'/kaggle/input/\') and not str(path).startswith(\'/kaggle/input/models/\'):\n        path=str(stage_checkpoint(path,max_bytes=32*2**30))\n    return _original_auto_loader(path,*args,**kwargs)\nAutoModel.from_pretrained=_staged_auto_loader\n\n"""Complementary frozen-feature stack (public Apache-2.0 notebooks by prvsiyan / lucasboesen / vnton37nguyn / kirilltym,\nas embedded in the public 0.6xx ATLAS notebooks). Run as a separate process; writes its raw log-probability blend."""\n# ============================================================ 0. configuration\nimport gc, glob, json, os, pathlib, re, time\nimport numpy as np, pandas as pd\ntry:\n    import torch\nexcept ImportError:   # local smoke without torch\n    torch = None\nfrom scipy.cluster.hierarchy import fcluster, linkage\nfrom scipy.sparse import csr_matrix, diags, hstack, vstack\nfrom scipy.spatial.distance import squareform\nfrom sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.preprocessing import StandardScaler\nfrom sklearn.svm import LinearSVC\n\nSMOKE = os.environ.get(\'NTL_SMOKE\') == \'1\'   # local code-path check: pseudo features instead of checkpoints, CPU\nT0 = time.time()\nLABELS = sorted([\'gpt-5.5\', \'gpt-5.6-luna\', \'gpt-5.6-sol\', \'gpt-5.6-terra\', \'gpt-6-astra\',\n                 \'gemini-3.1-pro-preview\', \'gemini-3.6-flash\', \'gemini-3.7-flash\', \'gemini-3.8-flash\',\n                 \'claude-fable-5\', \'claude-opus-4-6\', \'claude-opus-4-8\', \'claude-opus-5\', \'claude-sonnet-5\',\n                 \'qwen3.8-max\', \'glm-5.3\', \'glm-5.3-flash\', \'kimi-k3\'])\nK = len(LABELS)\nLAB2I = {l: i for i, l in enumerate(LABELS)}\nFAMILY_IDX = {f: np.array([i for i, l in enumerate(LABELS) if l.startswith(f)]) for f in (\'claude\', \'gpt\', \'gemini\', \'glm\')}\n\n# ---- test-pool grouping\nGROUP_THRESH, GROUP_LAYER, GROUP_MIN, GROUP_MAX = 0.60, 24, 2, 18   # mixed content + GLM-L24 distance, average linkage\nSTYLE_GROUP_THRESH = 0.90                                           # stylegram\'s own median-normalised content distance\n\n# ---- blend weights: log-probability multipliers of the 0.563 notebook (its L-BFGS fit on prompt-grouped OOF)\nW = {\'stylegram\': 1.516, \'stylometry_lgb\': 0.063, \'ultrabert\': 0.095,\n     \'q3_8b\': 0.0, \'q25_7b\': 0.0, \'q3_4b\': 0.0, \'glm9b\': 0.0, \'debv3l\': 0.05, \'electl\': 0.032, \'desk\': 0.132,\n     \'lora24_public\': 0.177, \'lora24_seed7\': 0.113, \'lora24_claude\': 0.177, \'lora20_q25_instyle\': 0.17,\n     \'tokattn_q35\': 0.153, \'tokattn_deb\': 0.185}\nTWIN_W = {\'q3_8b\': 1.5 * 0.219, \'q25_7b\': 1.5 * 0.141, \'q3_4b\': 1.5 * 0.083, \'glm9b\': 1.5 * 0.110, \'debv3l\': 0.25}\nLORA_TWIN_W = {\'lora24_public\': 0.15, \'lora24_claude\': 0.15}\nGATE_LAM = {\'claude_style\': 1.5, \'claude_lora\': 1.0, \'family\': 0.25}\nBALANCE_ALPHA = 0.5\nEXTRA_MIN_LABELS, EXTRA_SHARE = 12, 0.5\n\n# ---- the addition: shared-prompt Gaussian group heads on four measured views, weight = the centred twin of the same backbone,\n#      reduction / temperatures from our production fit (research/gaussian/production_weights_v8g.json, 135 training prompts)\nGAUSS_HEADS = {\'gauss_q3_8b_L22\': dict(src=\'q3_8b\', part=\'L22_mean\', d=384, T1=2.351, Tg=2.306, w=TWIN_W[\'q3_8b\']),\n               \'gauss_glm9b_L8\': dict(src=\'glm9b\', part=\'L8_mean\', d=384, T1=2.722, Tg=2.596, w=TWIN_W[\'glm9b\']),\n               \'gauss_q3_4b_L4\': dict(src=\'q3_4b\', part=\'L4_mean\', d=384, T1=2.725, Tg=2.578, w=TWIN_W[\'q3_4b\']),\n               \'gauss_debv3l_L24\': dict(src=\'debv3l\', part=\'L24_mean\', d=384, T1=3.112, Tg=3.069, w=TWIN_W[\'debv3l\'])}\nGAUSS_SETTING = dict(red=\'wpca\', se_shrink=\'lw\', sp_shrink=\'lw\', lam_floor=1e-4, n_exact=3, inj=True, iters=5, repel=True, sp_scale=1.0)\n\n# ---- member specifications\nLM_HEADS = [  # key, model_type, hidden size, parts of the plain head [(layer, pool)], extra parts extracted in the same pass, C\n    (\'glm9b\', \'glm4\', 4096, [(8, \'mean\'), (3, \'mean\')], [(GROUP_LAYER, \'mean\')], 0.0025),   # first: its L24 builds the groups\n    (\'q3_8b\', \'qwen3\', 4096, [(7, \'mean\'), (22, \'mean\')], [], 0.0025),\n    (\'q25_7b\', \'qwen2\', 3584, [(6, \'mean\'), (20, \'mean\')], [], 0.0025),\n    (\'q3_4b\', \'qwen3\', 2560, [(4, \'style\'), (7, \'style\')], [(4, \'mean\')], 0.0025)]\nENC_HEADS = [  # key, input directory name, parts, C\n    (\'debv3l\', \'deberta-v3-large\', [(24, \'mean\'), (6, \'mean\')], 0.005),\n    (\'electl\', \'electra-large-discriminator\', [(21, \'mean\'), (24, \'style\')], 0.005),\n    (\'desk\', \'desklib-detector\', [(23, \'mean\'), (23, \'style\')], 0.005)]\nLORA_RUNS = [  # (trunk model_type, hidden, kept layers), pooled layers, [(key, checkpoint marker, norms file, anchor features file)]\n    ((\'qwen3\', 4096, 24), [2, 4, 6, 8, 12, 16, 20, 24], [\n        (\'lora24_public\', \'**/lora24/final/COMPLETE.json\', \'heads.safetensors\', \'anchor_features.npz\'),\n        (\'lora24_seed7\', \'**/name-that-llm-train-lora24-s7/lora_out/job_final/COMPLETE.json\', \'norms.safetensors\', \'valid_features.npz\'),\n        (\'lora24_claude\', \'**/ntl-qwen3-8b-lora-claude-head/**/final/COMPLETE.json\', \'heads.safetensors\', \'anchor_features.npz\')]),\n    ((\'qwen2\', 3584, 20), [2, 4, 6, 8, 10, 12, 16, 20], [\n        (\'lora20_q25_instyle\', \'**/ntl-train-lora-qwen25-instyle/lora_out/job_final/COMPLETE.json\', \'norms.safetensors\', \'valid_features.npz\')])]\nLORA_MAX_LEN, LORA_BS = 447, 8\nTOKATTN_HEADS = [(\'tokattn_q35\', \'tokattn_final.pt\', \'qwen3_5\'), (\'tokattn_deb\', \'tokattn_debl23_final.pt\', \'deberta\')]\n\n# ---- data\nif SMOKE:\n    _all = pd.read_csv(os.environ[\'NTL_DATA\'] + \'/train.csv\')\n    _p = _all[\'prompt_id\'].drop_duplicates().sample(frac=1, random_state=0).tolist()\n    train = _all[_all[\'prompt_id\'].isin(_p[:30])].reset_index(drop=True)\n    test = _all[_all[\'prompt_id\'].isin(_p[30:42])].sample(frac=1, random_state=1).reset_index(drop=True)\n    test = pd.concat([test[test[\'prompt_id\'] == p].head(k) for p, k in zip(_p[30:42], [1, 1, 2, 2, 2, 3, 3, 4, 6, 9, 18, 18])]).reset_index(drop=True)\n    if os.environ.get(\'NTL_TINY\') == \'1\':\n        test = test.head(3).reset_index(drop=True)\n    SMOKE_Y = test[\'model\'].map(LAB2I).to_numpy(dtype=int)\nelse:\n    DATA_DIR = next(p for p in [pathlib.Path(\'/kaggle/input/competitions/name-that-llm\'), pathlib.Path(\'/kaggle/input/name-that-llm\')]\n                    if (p / \'train.csv\').exists())\n    train, test = pd.read_csv(DATA_DIR / \'train.csv\'), pd.read_csv(DATA_DIR / \'test.csv\')\n    if os.environ.get(\'NTL_FAKE_TEST\'):   # timing probe: pool passages as a stand-in test (same rows/seed as infer.py --fake_test)\n        test = pd.read_parquet(os.environ[\'NTL_POOL\']).sample(int(os.environ[\'NTL_FAKE_TEST\']), random_state=0)[[\'text\']].reset_index(drop=True)\n        test[\'id\'] = [f\'fake{i}\' for i in range(len(test))]\ntest[\'text\'] = test[\'text\'].fillna(\'\').astype(str)\nTR, TE = train[\'text\'].tolist(), test[\'text\'].tolist()\ny = train[\'model\'].map(LAB2I).to_numpy(dtype=int)\nPROMPT = train[\'prompt_id\'].to_numpy(dtype=object)\nTINY = len(TE) < 50   # the interactive 3-row test: exercise the group paths on training rows so the hidden rerun cannot hit untested code\nDEV = \'cuda\' if torch is not None and torch.cuda.is_available() else \'cpu\'\nprint(f\'train {train.shape}  test {test.shape}  gpus {torch.cuda.device_count() if torch is not None else 0}  smoke {SMOKE}\', flush=True)\n\nMEMBER_EVIDENCE = {}\nLOGIT = np.zeros((len(TE), K))           # the running log-probability blend\n\n\ndef add_member(key, prob, rows=None, weight=None):\n    """Add weight * log(prob) to the blend (all rows, or the given row subset)."""\n    w = W[key] if weight is None else weight\n    lp = np.log(np.clip(prob, 1e-9, 1.0))\n    full=np.zeros((len(TE),K),dtype=np.float64)\n    if rows is None: full[:]=w*lp\n    else: full[rows]=w*lp\n    MEMBER_EVIDENCE[key]=full\n\n    if rows is None:\n        LOGIT[:] += w * lp\n    else:\n        LOGIT[rows] += w * lp\n    print(f\'  + {key:20s} weight {w:.3f} on {len(TE) if rows is None else int(np.sum(rows))} rows  ({time.time() - T0:.0f}s)\', flush=True)\n\n\ndef softmax_rows(S):\n    S = S - S.max(1, keepdims=True)\n    P = np.exp(S)\n    return P / P.sum(1, keepdims=True)\n\n\ndef full_prob(P_part, classes):\n    """Expand a (n, len(classes)) probability block to (n, K) on the global label order."""\n    P = np.zeros((len(P_part), K))\n    P[:, classes] = P_part\n    return P\n\n\ndef group_centre_dense(X, groups):\n    X = np.asarray(X, dtype=np.float32).copy()\n    for g in np.unique(groups):\n        i = groups == g\n        X[i] -= X[i].mean(0)\n    return X\n\n\ndef self_test_rows():\n    """Training rows of the first three prompts, grouped by their true prompt (used only when the test file is tiny)."""\n    idx = np.where(np.isin(PROMPT, pd.unique(PROMPT)[:3]))[0]\n    return idx, np.unique(PROMPT[idx], return_inverse=True)[1]\n\n# ============================================================ 1. grouping the test pool\ndef average_linkage_groups(D, thresh):\n    """Cluster ids (1-based) from a square distance matrix, average linkage, cut at thresh."""\n    if len(D) < 2:\n        return np.ones(len(D), dtype=int)\n    D = np.asarray(D, dtype=np.float64).copy()\n    np.fill_diagonal(D, 0.0)\n    return fcluster(linkage(squareform(D, checks=False), method=\'average\'), thresh, criterion=\'distance\')\n\n\ndef content_cosine_distance(texts, max_features=None):\n    X = TfidfVectorizer(stop_words=\'english\', sublinear_tf=True, max_features=max_features, dtype=np.float32).fit_transform(texts)\n    return np.clip(1.0 - (X @ X.T).toarray(), 0.0, 1.0)\n\n\ndef mixed_groups(texts, E, thresh=GROUP_THRESH):\n    """Average of the content-word TF-IDF cosine distance and the cosine distance of mean-centred, L2-normalised embeddings E."""\n    if len(texts) < 2:\n        return np.ones(len(texts), dtype=int)\n    Z = np.asarray(E, dtype=np.float64)\n    Z = Z - Z.mean(0)\n    Z /= np.linalg.norm(Z, axis=1, keepdims=True) + 1e-12\n    D = 0.5 * (content_cosine_distance(texts) + np.clip(1.0 - Z @ Z.T, 0.0, 2.0))\n    return average_linkage_groups(D, thresh)\n\n\ndef normalised_content_groups(texts, thresh=STYLE_GROUP_THRESH):\n    """The stylegram\'s grouping: content distance divided by its median off-diagonal value, cut at thresh."""\n    if len(texts) < 2:\n        return np.ones(len(texts), dtype=int)\n    D = content_cosine_distance(texts, max_features=100_000)\n    med = max(float(np.median(D[np.triu_indices(len(texts), 1)])), 1e-6)\n    return average_linkage_groups(D / med, thresh)\n\n\ndef eligible(groups, lo=GROUP_MIN, hi=GROUP_MAX):\n    size = np.bincount(groups)[groups]\n    return (size >= lo) & (size <= hi)\n\n\ndef describe_groups(name, groups):\n    size = np.bincount(groups)[groups]\n    print(f\'{name}: {len(groups)} rows | singletons {int((size == 1).sum())}  2-3: {int(((size >= 2) & (size <= 3)).sum())}\'\n          f\'  4-18: {int(((size >= 4) & (size <= 18)).sum())}  >18: {int((size > 18).sum())}\', flush=True)\n\n\nG_TE, GROUP_ROWS = None, None   # set after the GLM pass (section 4)\n\n# ============================================================ 2. style n-grams + extra passages + gates\nFUNCTION_WORDS = frozenset("a an the and or but if while although because as so than that which who whom whose this these those it its "\n                           "they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with "\n                           "without for of by over under between through into about is are was were be been being have has had do does did "\n                           "can could will would shall should may might must not no nor all any some each every both either neither other "\n                           "another such only also just very too more most less least here there when where why how what whether then now "\n                           "still yet even much many few little".split())\nTOKEN_RE = re.compile(r"\\w+|[^\\w\\s]")\n\n\ndef shape_view(text):\n    return re.sub(r"[^\\W\\d_]+", lambda m: \'X\' if m[0].isupper() else \'x\', text)\n\n\ndef function_view(text):\n    return \' \'.join(t.lower() if (t.lower() in FUNCTION_WORDS or not t.isalnum()) else \'CONTENT\' for t in TOKEN_RE.findall(text))\n\n\nclass StyleGrams:\n    """Four stacked TF-IDF views: char 3-5, word 1-3, shape (char 2-6 of the x/X text), function-word (word 1-3 of the masked text)."""\n    VIEWS = [(\'char\', str, (3, 5), 250_000, 1.0), (\'word\', str, (1, 3), 180_000, 0.7),\n             (\'char\', shape_view, (2, 6), 60_000, 0.5), (\'word\', function_view, (1, 3), 50_000, 0.5)]\n\n    def fit(self, texts):\n        self.vecs = []\n        for analyzer, view, ngr, maxf, w in self.VIEWS:\n            opts = dict(analyzer=analyzer, ngram_range=ngr, lowercase=False, sublinear_tf=True, min_df=2, max_features=maxf, dtype=np.float64)\n            if analyzer == \'word\':\n                opts[\'token_pattern\'] = r"(?u)\\b\\w+\\b|[^\\w\\s]"\n            self.vecs.append((TfidfVectorizer(**opts).fit([view(t) for t in texts]), view, w))\n        return self\n\n    def transform(self, texts):\n        return hstack([v.transform([view(t) for t in texts]) * w for v, view, w in self.vecs], format=\'csr\')\n\n\ndef sparse_group_centre(X, groups, lo=GROUP_MIN, hi=GROUP_MAX):\n    """Subtract each group\'s mean row (sparse); groups outside [lo, hi] are left as they are."""\n    X = csr_matrix(X)\n    _, inv, cnt = np.unique(np.asarray(groups), return_inverse=True, return_counts=True)\n    keep = np.flatnonzero((cnt >= lo) & (cnt <= hi))\n    if not len(keep):\n        return X.copy()\n    col = np.full(len(cnt), -1); col[keep] = np.arange(len(keep))\n    rows = np.flatnonzero(col[inv] >= 0)\n    M = csr_matrix((np.ones(len(rows)), (rows, col[inv][rows])), shape=(X.shape[0], len(keep)))\n    means = diags(1.0 / cnt[keep]) @ (M.T @ X)\n    return (X - M @ means).tocsr()\n\n\ndef svc_probs(svc, X):\n    return full_prob(softmax_rows(svc.decision_function(X)), svc.classes_)\n\n\n# ---- public in-style passages (prompt-bank protocol), one per (prompt, label); prompts answered by >= EXTRA_MIN_LABELS labels\ndef load_extra_passages():\n    sources = [(\'**/ntl_extra_passages_claude.csv\', \'model\', None), (\'**/gen_passages.csv\', \'model\', None),\n               (\'**/ntl_generations.csv\', \'label\', \'finish_reason\'), (\'**/ntl_proxy_generations.csv\', \'label\', None),\n               (\'**/ntl_generated*.csv\', \'model\', \'finish\'), (\'**/passages_clean.csv\', \'model\', None)]\n    parts = []\n    for pattern, label_col, finish_col in sources:\n        for f in sorted(glob.glob(\'/kaggle/input/\' + pattern, recursive=True)):\n            d = pd.read_csv(f)\n            if finish_col and finish_col in d:\n                d = d[d[finish_col].eq(\'stop\')]\n            d = d.rename(columns={label_col: \'model\'})[[\'prompt_id\', \'model\', \'text\']].copy()\n            d[\'prompt_id\'] = (\'pv_\' if f.endswith(\'passages_clean.csv\') else \'\') + d[\'prompt_id\'].astype(str)   # prvsiyan\'s prompts are a different bank\n            parts.append(d)\n    if not parts:\n        return pd.DataFrame(columns=[\'prompt_id\', \'model\', \'text\'])\n    x = pd.concat(parts, ignore_index=True)\n    x = x[x[\'model\'].isin(LABELS)].copy()\n    x[\'text\'] = x[\'text\'].astype(str).map(lambda t: re.sub(r\'\\s+\', \' \', re.sub(r\'^\\*\\*(.*)\\*\\*$\', r\'\\1\', t.strip())).strip())\n    x = x[x[\'text\'].str.split().str.len().between(30, 300)].drop_duplicates(subset=[\'model\', \'text\'])\n    x = x.drop_duplicates(subset=[\'prompt_id\', \'model\'])\n    x = x[x.groupby(\'prompt_id\')[\'model\'].transform(\'nunique\') >= EXTRA_MIN_LABELS].reset_index(drop=True)\n    return x\n\n\nEXTRA = load_extra_passages()\nEX_T, EX_Y, EX_G = EXTRA[\'text\'].tolist(), EXTRA[\'model\'].map(LAB2I).to_numpy(dtype=int), EXTRA[\'prompt_id\'].to_numpy(dtype=object)\nprint(f\'extra passages: {len(EXTRA)} rows, {EXTRA["prompt_id"].nunique()} prompts\', flush=True)\n\n\ndef extra_sample_weights():\n    """Per class: original rows keep (1 - EXTRA_SHARE) of the class weight, the class\'s extra rows share EXTRA_SHARE of it."""\n    n_tr, n_ex = np.bincount(y, minlength=K).astype(float), np.bincount(EX_Y, minlength=K).astype(float)\n    w_tr = np.where(n_ex[y] > 0, 1.0 - EXTRA_SHARE, 1.0)\n    w_ex = EXTRA_SHARE * n_tr[EX_Y] / np.maximum(n_ex[EX_Y], 1.0)\n    return np.concatenate([w_tr, w_ex])\n\n\ngrams = StyleGrams().fit(TR + EX_T)                       # vocabulary and idf from train + extra\nX_tr, X_ex, X_te = grams.transform(TR), grams.transform(EX_T) if len(EX_T) else None, grams.transform(TE)\n\n\ndef fit_svc(X, labels, C=0.5, **kw):\n    return LinearSVC(C=C, random_state=42, max_iter=4000).fit(X, labels, **kw)\n\n\nif X_ex is not None:\n    svc_plain = fit_svc(vstack([X_tr, X_ex]).tocsr(), np.concatenate([y, EX_Y]), sample_weight=extra_sample_weights())\nelse:\n    svc_plain = fit_svc(X_tr, y)\nP_style = svc_probs(svc_plain, X_te)\n\n# centred twin: prompt-centred training rows (extra rows centred within their own prompts) -> inferred test groups\nX_tr_c = sparse_group_centre(X_tr, PROMPT)\nif X_ex is not None:\n    svc_centred = fit_svc(vstack([X_tr_c, sparse_group_centre(X_ex, EX_G, GROUP_MIN, 10 ** 6)]).tocsr(), np.concatenate([y, EX_Y]),\n                          sample_weight=extra_sample_weights())\nelse:\n    svc_centred = fit_svc(X_tr_c, y)\nG_style = normalised_content_groups(TE)\nrows_style = eligible(G_style)\ndescribe_groups(\'stylegram groups\', G_style)\nif rows_style.any():\n    P_style[rows_style] = svc_probs(svc_centred, sparse_group_centre(X_te[rows_style], G_style[rows_style]))\nif TINY:   # exercise the centred path on training rows grouped by their true prompt (output discarded)\n    st_idx, st_g = self_test_rows()\n    acc = np.mean(svc_centred.classes_[svc_centred.decision_function(sparse_group_centre(X_tr[st_idx], st_g)).argmax(1)] == y[st_idx])\n    print(f\'  centred stylegram self-test on {len(st_idx)} training rows (seen in training): acc {acc:.2f}\', flush=True)\nadd_member(\'stylegram\', P_style)\ndel X_tr_c, svc_plain, svc_centred, P_style\n\n\n# ---- gates: style SVCs that only re-rank inside one family after the blend\ndef family_gate(texts, labels, classes, C, sample_weight=None, balanced=False):\n    g = StyleGrams().fit(texts)\n    svc = LinearSVC(C=C, class_weight=\'balanced\' if balanced else None, random_state=42, max_iter=4000).fit(\n        g.transform(texts), labels, sample_weight=sample_weight)\n    assert list(svc.classes_) == list(classes)\n    d = svc.decision_function(g.transform(TE))\n    return np.column_stack([-d, d]) if len(classes) == 2 else d\n\n\nGATES = {}\napi_claude = sorted(glob.glob(\'/kaggle/input/**/ntl_extra_passages_claude.csv\', recursive=True))\nif api_claude:\n    api = pd.read_csv(api_claude[0])\n    m = np.isin(y, FAMILY_IDX[\'claude\'])\n    GATES[\'claude\'] = family_gate([t for t, k in zip(TR, m) if k] + api[\'text\'].astype(str).tolist(),\n                                  np.concatenate([y[m], api[\'model\'].map(LAB2I).to_numpy(dtype=int)]), FAMILY_IDX[\'claude\'], 0.5)\n    print(f\'claude gate: {int(m.sum())} official + {len(api)} API rows\', flush=True)\n    del api\nelse:\n    print(\'Claude API passages not attached: Claude gate off\', flush=True)\n# The gpt / gemini / glm family gates of the upstream stack were computed but never used downstream: removed (runtime only).\ndel grams, X_tr, X_ex, X_te\n\n# ============================================================ 3. stylometric features + LightGBM\nPUNCT = list(\',.;:!?()[]"\\\'-\') + [\'—\', \'–\', \'‘\', \'’\', \'“\', \'”\', \'…\', \'*\', \'/\', \'%\']\nSTOPS = sorted(ENGLISH_STOP_WORDS)\nSENT_RE, WORD_RE = re.compile(r\'(?<=[.!?])\\s+\'), re.compile(r"[A-Za-z’\']+")\n\n\ndef stylometry(t):\n    words = WORD_RE.findall(t)\n    low = [w.lower() for w in words]\n    nw, nc = max(len(words), 1), max(len(t), 1)\n    sents = [s for s in SENT_RE.split(t.strip()) if s]\n    sl = np.array([len(WORD_RE.findall(s)) for s in sents] or [0], dtype=float)\n    wl = np.array([len(w) for w in words] or [0], dtype=float)\n    f = [np.log1p(nw), len(sents), sl.mean(), sl.std(), sl.max(), sl.min(), wl.mean(), wl.std(), (wl >= 9).mean(), len(set(low)) / nw,\n         sum(c.isupper() for c in t) / nc, sum(c.isdigit() for c in t) / nc, sum(ord(c) > 127 for c in t) / nc,\n         t.count(\', \') / nw, t.count(\' and \') / nw, t.count(\' but \') / nw,\n         sum(1 for s in sents if s[:1].isupper() and s.split()[0].lower() in (\'this\', \'these\', \'it\', \'they\')) / max(len(sents), 1),\n         sum(w.endswith(\'ly\') for w in words) / nw, sum(w.endswith(\'ing\') for w in words) / nw, sum(w.endswith(\'tion\') for w in words) / nw]\n    f += [t.count(q) / nw for q in PUNCT]\n    counts = pd.Series(low).value_counts() if low else pd.Series(dtype=float)\n    f += [counts.get(s, 0) / nw for s in STOPS]\n    return f\n\n\nF_tr, F_te = (np.array([stylometry(t) for t in txt], dtype=np.float32) for txt in (TR, TE))\ntry:\n    import lightgbm as lgb\n    booster = lgb.LGBMClassifier(n_estimators=400, learning_rate=0.03, num_leaves=15, min_child_samples=10, subsample=0.8, subsample_freq=1,\n                                 colsample_bytree=0.5, reg_lambda=1.0, random_state=0, verbose=-1, n_jobs=4).fit(F_tr, y)\n    add_member(\'stylometry_lgb\', full_prob(booster.predict_proba(F_te), booster.classes_))\n    del booster\nexcept ImportError:\n    assert SMOKE, \'lightgbm missing\'\n    print(\'smoke: lightgbm unavailable, stylometry member skipped\', flush=True)\ndel F_tr, F_te\n\n# ============================================================ 4b. shared-prompt Gaussian group head\n# Verbatim core of our research tool tools/group_gaussian.py (sha256 8a42427b77a37f56...), numpy only.\n# Model per dense view, in a reduced space z (standardise -> within-prompt-residual PCA, fitted on training rows):\n#   z_i = mu_{y_i} + p_g + e_i,  p_g ~ N(0, S_p) shared by every passage of prompt group g,  e_i ~ N(0, S_e).\n# A joint diagonalisation makes the dimensions independent; for a detected group of n passages the exact joint\n# log-likelihood of a label tuple is sum_i a_i(c_i) + sum_{i<j} b_ij(c_i, c_j). Groups of 2-3 are decoded exactly over\n# all label tuples with the at-most-once constraint; larger groups by a cavity EM with the finite-population-corrected\n# prompt-effect estimate lam / (1 + (n-1) lam); singletons integrate the prompt effect out (never used here).\nimport itertools\nK = 18\nDEFAULT = {"red": "wpca", "d": 256, "se_shrink": "lw", "sp_shrink": "lw", "lam_floor": 1e-4, "n_exact": 3, "inj": True,\n           "iters": 5, "repel": True, "sp_scale": 1.0}\n\n# =================================================================== primitives ====\ndef _lse(A, axis=None, keepdims=False):\n    m = np.max(A, axis=axis, keepdims=True); m = np.where(np.isfinite(m), m, 0.0)\n    out = np.log(np.sum(np.exp(A - m), axis=axis, keepdims=True)) + m\n    if keepdims: return out\n    return np.squeeze(out, axis=axis) if axis is not None else float(out.item())\n\ndef log_softmax(L): return L - _lse(L, axis=1, keepdims=True)\n\ndef _lw_shrinkage(X):\n    """Ledoit-Wolf shrinkage coefficient for rows X that are already centred (sklearn formula, assume_centered)."""\n    from sklearn.covariance import ledoit_wolf_shrinkage\n    return float(ledoit_wolf_shrinkage(X, assume_centered=True))\n\ndef _shrunk(S, a):\n    d = S.shape[0]; return (1 - a) * S + a * (np.trace(S) / d) * np.eye(d)\n\ndef _two_way(Z, y, pinv, G, iters=4):\n    """additive fit Z = mu[y] + P[prompt] + E by alternating means (balanced design -> converges in 1-2 rounds)."""\n    P = np.zeros((G, Z.shape[1])); cnt_p = np.bincount(pinv, minlength=G).astype(float)\n    cls = np.unique(y); mu = np.zeros((K, Z.shape[1]))\n    for _ in range(iters):\n        R = Z - P[pinv]\n        for c in cls: mu[c] = R[y == c].mean(0)\n        R = Z - mu[y]; P = np.zeros_like(P); np.add.at(P, pinv, R); P /= cnt_p[:, None]\n    return mu, P, cnt_p\n\ndef fit(X, y, prompts, cfg=None):\n    """fit the shared-prompt Gaussian on training rows. X (N, D) raw features, y (N,) labels in 0..17, prompts (N,) group ids.\n    Returns a params dict (numpy arrays + floats); temperatures default to 1 (set by the caller\'s cross-fit)."""\n    c = dict(DEFAULT); c.update(cfg or {})\n    X = np.asarray(X, dtype=np.float64); y = np.asarray(y); N, D = X.shape\n    pu, pinv = np.unique(np.asarray(prompts), return_inverse=True); G = len(pu)\n    mx = X.mean(0); sx = X.std(0); sx = np.where(sx > 1e-8, sx, 1.0); Xs = (X - mx) / sx\n    if c["red"] == "none":\n        V = np.eye(D)\n    else:\n        if c["red"] == "wpca":\n            Pm = np.zeros((G, D)); np.add.at(Pm, pinv, Xs); Pm /= np.bincount(pinv)[:, None]; R = Xs - Pm[pinv]\n        elif c["red"] == "pca": R = Xs\n        else: raise ValueError(c["red"])\n        d = int(min(c["d"], N - G - 1, D))\n        # economy SVD through the (N x N) Gram matrix (N << D for the real views)\n        if N < D:\n            w, U = np.linalg.eigh(R @ R.T); o = np.argsort(w)[::-1][:d]; w, U = np.clip(w[o], 1e-12, None), U[:, o]\n            V = (R.T @ U) / np.sqrt(w)\n        else:\n            _, _, Vt = np.linalg.svd(R, full_matrices=False); V = Vt[:d].T\n    Z = Xs @ V; d = Z.shape[1]\n    mu, P, ng = _two_way(Z, y, pinv, G)\n    E = Z - mu[y] - P[pinv]; dof = max(N - G - len(np.unique(y)) + 1, 1)\n    Se_raw = E.T @ E / dof\n    ae = _lw_shrinkage(E) if c["se_shrink"] == "lw" else float(c["se_shrink"]); Se = _shrunk(Se_raw, ae)\n    Pc = P - P.mean(0); Sp_raw = Pc.T @ Pc / max(G - 1, 1) - np.mean(1.0 / ng) * Se_raw\n    ap = _lw_shrinkage(Pc) if c["sp_shrink"] == "lw" else float(c["sp_shrink"])\n    tr = max(np.trace(Sp_raw), 1e-8 * np.trace(Se)); Sp = (1 - ap) * Sp_raw + ap * (tr / d) * np.eye(d); Sp *= c["sp_scale"]\n    # joint diagonalisation: A Se A\' = I, A Sp A\' = diag(lam)\n    L = np.linalg.cholesky(Se); Li = np.linalg.inv(L); M = Li @ Sp @ Li.T; M = 0.5 * (M + M.T)\n    lam, Q = np.linalg.eigh(M); lam = np.clip(lam, c["lam_floor"], None); A = Q.T @ Li\n    W = V @ A.T\n    return {"mx": mx, "sx": sx, "W": W, "m": mu @ A.T, "lam": lam, "T1": 1.0, "Tg": 1.0,\n            "info": {"d": int(d), "a_e": round(ae, 4), "a_p": round(ap, 4), "lam_pct": np.round(np.percentile(lam, [5, 50, 95, 100]), 3).tolist()}, "cfg": c}\n\ndef transform(X, prm): return ((np.asarray(X, dtype=np.float64) - prm["mx"]) / prm["sx"]) @ prm["W"]\n\n_MASK = {}\ndef _inj_mask(n, C):\n    if (n, C) not in _MASK:\n        grids = np.meshgrid(*[np.arange(C)] * n, indexing="ij"); bad = np.zeros([C] * n, bool)\n        for i, j in itertools.combinations(range(n), 2): bad |= grids[i] == grids[j]\n        _MASK[(n, C)] = bad\n    return _MASK[(n, C)]\n\ndef singleton_ll(U, m, lam):\n    """(n, C) log N(u; m_c, I + diag lam) up to a constant (p integrated out)."""\n    R = U[:, None, :] - m[None]; return -0.5 * (R ** 2 / (1 + lam)).sum(-1)\n\ndef joint_terms(U, m, lam):\n    """a (n, C) and B (n, n, C, C) of the exact joint log-likelihood of one group (constant in c dropped)."""\n    n = len(U); gam = lam / (1 + n * lam); R = U[:, None, :] - m[None]\n    a = -0.5 * (R ** 2 * (1 - gam)).sum(-1); G = R * np.sqrt(gam); B = np.einsum("icd,jed->ijce", G, G)\n    return a, B\n\ndef joint_table(a, B):\n    """full tuple table Q[c_1, ..., c_n] = sum_i a_i + sum_{i<j} B_ij."""\n    n, C = a.shape; Q = np.zeros([C] * n)\n    for i in range(n):\n        sh = [1] * n; sh[i] = C; Q = Q + a[i].reshape(sh)\n    for i, j in itertools.combinations(range(n), 2):\n        sh = [1] * n; sh[i] = C; sh[j] = C; Q = Q + B[i, j].reshape(sh)\n    return Q\n\ndef exact_marginals(Q, T=1.0, inj=True):\n    """per-passage log marginal posteriors (n, C) from a tuple table Q (unnormalised log joint), tempered by 1/T."""\n    n = Q.ndim; C = Q.shape[0]; Qt = Q / T\n    if inj and n >= 2: Qt = np.where(_inj_mask(n, C), -np.inf, Qt)\n    lz = _lse(Qt); out = np.empty((n, C))\n    for i in range(n):\n        ax = tuple(k for k in range(n) if k != i); out[i] = (_lse(Qt, axis=ax) if ax else Qt) - lz\n    return out\n\ndef em_marginals(U, m, lam, T=1.0, iters=5, repel=False, q0=None):\n    """cavity EM for one group (any n >= 2): returns (n, C) log q."""\n    n = len(U); v = lam / (1 + (n - 1) * lam)\n    L = singleton_ll(U, m, lam) / T if q0 is None else np.log(np.clip(q0, 1e-300, None))\n    q = np.exp(log_softmax(L))\n    for _ in range(iters):\n        s = U - q @ m; S = s.sum(0); Pm = (S[None] - s) * v                  # cavity prompt-effect means (n, d)\n        R = U[:, None, :] - Pm[:, None, :] - m[None]; L = -0.5 * (R ** 2 / (1 + v)).sum(-1) / T\n        if repel:   # mean-field at-most-once: discourage classes the other members already hold\n            lo = np.log(np.clip(1 - q, 1e-6, None)); L = L + (lo.sum(0)[None] - lo)\n        q = np.exp(log_softmax(L))\n    return log_softmax(L)\n\nLAM_DIFF = 1e6\ndef lam_g(prm, cfg=None):\n    """lam used for groups: the fitted one, or ~infinity in the paired-difference ablation (cfg \'diff\'): then the joint likelihood\n    only sees within-group differences (x_i - x_j around mu_a - mu_b with covariance 2 S_e; triples: centred observations vs\n    centred prototypes) and the prompt term cancels exactly."""\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {})\n    return np.full_like(prm["lam"], LAM_DIFF) if c.get("diff") else prm["lam"]\n\ndef group_logpost(U, prm, T1=None, Tg=None, cfg=None):\n    c = dict(prm.get("cfg", DEFAULT)); c.update(cfg or {}); n = len(U); lg = lam_g(prm, cfg)\n    T1 = prm["T1"] if T1 is None else T1; Tg = prm["Tg"] if Tg is None else Tg\n    if n == 1: return log_softmax(singleton_ll(U, prm["m"], prm["lam"]) / T1)\n    if n <= c["n_exact"]:\n        a, B = joint_terms(U, prm["m"], lg); return exact_marginals(joint_table(a, B), Tg, c["inj"])\n    return em_marginals(U, prm["m"], lg, Tg, c["iters"], c["repel"])\n\ndef scores(X_pool, lab, prm, T1=None, Tg=None, cfg=None, U=None):\n    """(n, 18) log-probabilities for every pool row; lab = detected group labels of the pool rows (singletons allowed)."""\n    U = transform(X_pool, prm) if U is None else U; lab = np.asarray(lab); n = len(U); out = np.empty((n, prm["m"].shape[0]))\n    sizes = np.bincount(lab); s1 = sizes[lab] == 1\n    T1 = prm["T1"] if T1 is None else T1\n    if s1.any(): out[s1] = log_softmax(singleton_ll(U[s1], prm["m"], prm["lam"]) / T1)\n    for g in np.where(sizes >= 2)[0]:\n        ii = np.where(lab == g)[0]; out[ii] = group_logpost(U[ii], prm, T1, Tg, cfg)\n    return out\n\n# ============================================================ 4. frozen models: plain heads, centred twins, Gaussian group heads\nif not SMOKE:\n    from transformers import AutoConfig, AutoModel, AutoTokenizer\nCACHED_PARTS = {}   # key -> {part name: (train features, test features)}\n\n\ndef find_checkpoint(model_type, hidden=None, dir_name=None):\n    """A checkpoint directory under /kaggle/input by model_type (+ hidden size) or by directory name."""\n    pattern = f\'/kaggle/input/**/{dir_name}/config.json\' if dir_name else \'/kaggle/input/**/config.json\'\n    for cfg in sorted(glob.glob(pattern, recursive=True)):\n        try:\n            c = json.load(open(cfg))\n        except Exception:\n            continue\n        if dir_name or (c.get(\'model_type\') == model_type and (hidden is None or c.get(\'hidden_size\') == hidden)):\n            return os.path.dirname(cfg)\n    return None\n\n\ndef load_truncated_decoder(path, keep, dtype):\n    cfg = AutoConfig.from_pretrained(path)\n    cfg.num_hidden_layers = keep\n    if getattr(cfg, \'layer_types\', None):\n        cfg.layer_types = cfg.layer_types[:keep]\n    kw = dict(config=cfg, dtype=dtype)\n    if DEV == \'cuda\':\n        kw[\'device_map\'] = \'auto\'\n    return AutoModel.from_pretrained(path, **kw).eval()\n\n\ndef style_token_mask(tok):\n    """Vocabulary mask of \'style\' tokens: the decoded piece is an English stop word or pure punctuation."""\n    pieces = [tok.decode([i]) for i in range(len(tok))]\n    return torch.tensor([(p.strip().lower() in ENGLISH_STOP_WORDS) or bool(re.fullmatch(r\'\\s*[^\\w\\s]+\\s*\', p)) for p in pieces])\n\n\n@torch.no_grad()\ndef pooled_states(model, tok, texts, parts, decoder, bs, max_len=512):\n    """{\'L{layer}_{pool}\': (n, hidden)} masked mean pools of hidden_states[layer] (all tokens, or style tokens only)."""\n    H = model.config.hidden_size\n    out = {f\'L{L}_{pool}\': np.zeros((len(texts), H), dtype=np.float32) for L, pool in parts}\n    smask = style_token_mask(tok) if any(pool == \'style\' for _, pool in parts) else None\n    dev = next(model.parameters()).device\n    order = np.argsort([len(t) for t in texts])\n    retried = 0\n    for b0 in range(0, len(texts), bs):\n        idx = order[b0:b0 + bs]\n        if decoder:\n            batch = [(tok.bos_token or \'\') + texts[i] for i in idx]\n            enc = tok(batch, return_tensors=\'pt\', padding=True, truncation=True, max_length=max_len, add_special_tokens=False).to(dev)\n        else:\n            enc = tok([texts[i] for i in idx], return_tensors=\'pt\', padding=True, truncation=True, max_length=max_len).to(dev)\n        hs = model(**enc, output_hidden_states=True).hidden_states\n        if decoder and not all(torch.isfinite(hs[L]).all() for L, _ in parts):   # fp16 overflow: redo the batch in bf16\n            retried += 1\n            model.to(torch.bfloat16)\n            hs = model(**enc, output_hidden_states=True).hidden_states\n        m_all = enc[\'attention_mask\'].float()\n        for L, pool in parts:\n            m = m_all * smask.to(m_all.device)[enc[\'input_ids\']].float() if pool == \'style\' else m_all\n            m = m.unsqueeze(-1).to(hs[L].device)\n            out[f\'L{L}_{pool}\'][idx] = ((hs[L].float() * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()\n        del hs, enc\n    if retried:\n        print(f\'  {retried} batches redone in bf16\', flush=True)\n    return {k: np.nan_to_num(v) for k, v in out.items()}\n\n\ndef pseudo_parts(parts, dim=48):\n    """SMOKE only: deterministic stand-ins for the model passes (TF-IDF SVD of the texts), same downstream code."""\n    from sklearn.decomposition import TruncatedSVD\n    tf = TfidfVectorizer(analyzer=\'char\', ngram_range=(2, 4), sublinear_tf=True, min_df=2).fit(TR + TE)\n    out = {}\n    for j, (L, pool) in enumerate(parts):\n        svd = TruncatedSVD(dim, random_state=L + 7 * j).fit(tf.transform(TR))\n        out[f\'L{L}_{pool}\'] = (svd.transform(tf.transform(TR)).astype(np.float32), svd.transform(tf.transform(TE)).astype(np.float32))\n    return out\n\n\ndef extract_parts(key, parts, loader):\n    """Train and test pools for every requested part; loader() returns (model, tok, decoder flag, batch size)."""\n    if SMOKE:\n        return pseudo_parts(parts)\n    model, tok, decoder, bs = loader()\n    te = pooled_states(model, tok, TE, parts, decoder, bs)\n    group_only_unused = (key == \'glm9b\' and W[key] == 0 and\n        not eligible(mixed_groups(TE, te[f\'L{GROUP_LAYER}_mean\'])).any())\n    if group_only_unused:\n        tr = {k: np.zeros((len(TR), v.shape[1]), dtype=np.float32) for k,v in te.items()}\n        print(\'SKIP_IDENTICALLY_ZERO_GLM_TRAIN_HEADS_NO_ELIGIBLE_QUERY_GROUPS\',flush=True)\n    else:\n        tr = pooled_states(model, tok, TR, parts, decoder, bs)\n    if not group_only_unused:\n        cache_dir=pathlib.Path(\'/kaggle/working/complement_features\');cache_dir.mkdir(exist_ok=True)\n        np.savez_compressed(cache_dir/(key+\'.npz\'),\n            ids=train.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),\n            y=y,prompts=np.asarray(PROMPT,dtype=str),test_ids=test.id.to_numpy(dtype=str),\n            **{k+\'_train\':v for k,v in tr.items()},**{k+\'_test\':v for k,v in te.items()})\n    del model, tok\n    gc.collect()\n    if DEV == \'cuda\':\n        torch.cuda.empty_cache()\n    return {k: (tr[k], te[k]) for k in tr}\n\n\ndef decoder_loader(model_type, hidden, keep):\n    def load():\n        path = find_checkpoint(model_type, hidden)\n        assert path, f\'no {model_type} checkpoint with hidden size {hidden} under /kaggle/input\'\n        tok = AutoTokenizer.from_pretrained(path)\n        tok.padding_side = \'right\'\n        if tok.pad_token is None:\n            tok.pad_token = tok.eos_token\n        return load_truncated_decoder(path, keep, torch.float16 if DEV == \'cuda\' else torch.float32), tok, True, 8\n    return load\n\n\ndef encoder_loader(dir_name):\n    def load():\n        path = find_checkpoint(None, dir_name=dir_name)\n        assert path, f\'encoder directory {dir_name} not found under /kaggle/input\'\n        return AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval(), AutoTokenizer.from_pretrained(path), False, 16\n    return load\n\n\ndef concat_parts(cache, parts):\n    return (np.concatenate([cache[f\'L{L}_{p}\'][0] for L, p in parts], 1), np.concatenate([cache[f\'L{L}_{p}\'][1] for L, p in parts], 1))\n\n\ndef plain_and_twin(key, Etr, Ete, C):\n    """Plain head (test-side standardisation) for the blend; centred twin evidence on grouped rows."""\n    if W[key] > 0:\n        sc = StandardScaler().fit(Etr)\n        clf = LogisticRegression(C=C, max_iter=5000).fit(sc.transform(Etr), y)\n        Xte = (Ete - Ete.mean(0)) / (Ete.std(0) + 1e-6) if len(Ete) >= 2 else sc.transform(Ete)\n        add_member(key, full_prob(clf.predict_proba(Xte), clf.classes_))\n    else:\n        print(f\'  {key}: plain head has weight 0 in the 0.563 blend, not fitted\', flush=True)\n    if key in TWIN_W:\n        csc = StandardScaler().fit(group_centre_dense(Etr, PROMPT))\n        twin = LogisticRegression(C=C, max_iter=5000).fit(csc.transform(group_centre_dense(Etr, PROMPT)), y)\n        if GROUP_ROWS.any():\n            pc = twin.predict_proba(csc.transform(group_centre_dense(Ete[GROUP_ROWS], G_TE[GROUP_ROWS])))\n            add_member(key + \'_twin\', full_prob(pc, twin.classes_), rows=GROUP_ROWS, weight=TWIN_W[key])\n        if TINY:\n            st_idx, st_g = self_test_rows()\n            ps = twin.predict_proba(csc.transform(group_centre_dense(Etr[st_idx], st_g)))\n            print(f\'  {key} twin self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(twin.classes_[ps.argmax(1)] == y[st_idx]):.2f}\', flush=True)\n\n\ndef gaussian_group_head(name, spec, Xtr, Xte):\n    """Our addition: fit the shared-prompt Gaussian on the training prompts, score the grouped test rows jointly within their group."""\n    prm = fit(Xtr, y, PROMPT, dict(GAUSS_SETTING, d=spec[\'d\']))\n    prm[\'T1\'], prm[\'Tg\'] = spec[\'T1\'], spec[\'Tg\']\n    print(f\'  {name}: d {prm["info"]["d"]}, shrinkage e/p {prm["info"]["a_e"]}/{prm["info"]["a_p"]}, lam pct {prm["info"]["lam_pct"]}\', flush=True)\n    if GROUP_ROWS.any():\n        LP = scores(Xte[GROUP_ROWS], np.unique(G_TE[GROUP_ROWS], return_inverse=True)[1], prm)\n        add_member(name, np.exp(LP), rows=GROUP_ROWS, weight=spec[\'w\'])\n    if TINY:\n        st_idx, st_g = self_test_rows()\n        LP = scores(Xtr[st_idx], st_g, prm)\n        print(f\'  {name} self-test on {len(st_idx)} training rows (seen in training): acc {np.mean(LP.argmax(1) == y[st_idx]):.2f}\', flush=True)\n\n\nfor key, model_type, hidden, parts, extra_parts, C in LM_HEADS:\n    if G_TE is not None and not GROUP_ROWS.any() and W[key] == 0:\n        print(\'SKIP_IDENTICALLY_ZERO_QUERY_EXPERT\',key,flush=True)\n        continue\n    print(f\'[{key}] {model_type} hidden {hidden} parts {parts + extra_parts}\', flush=True)\n    cache = extract_parts(key, parts + extra_parts, decoder_loader(model_type, hidden, max(L for L, _ in parts + extra_parts) + 1))\n    if G_TE is None:   # the GLM pass comes first: its layer-24 pool defines the test groups\n        G_TE = mixed_groups(TE, cache[f\'L{GROUP_LAYER}_mean\'][1])\n        GROUP_ROWS = eligible(G_TE)\n        describe_groups(\'test groups (content + GLM L24)\', G_TE)\n        if SMOKE:\n            same_g, same_p = G_TE[:, None] == G_TE[None, :], test[\'prompt_id\'].to_numpy(dtype=object)[:, None] == test[\'prompt_id\'].to_numpy(dtype=object)[None, :]\n            off = ~np.eye(len(TE), dtype=bool)\n            print(f\'  smoke grouping: pair precision {(same_g & same_p & off).sum() / max((same_g & off).sum(), 1):.2f} \'\n                  f\'recall {(same_g & same_p & off).sum() / max((same_p & off).sum(), 1):.2f}\', flush=True)\n    if not GROUP_ROWS.any() and W[key] == 0:\n        del cache\n        continue\n    CACHED_PARTS[key] = cache\n    plain_and_twin(key, *concat_parts(cache, parts), C)\n    for name, spec in GAUSS_HEADS.items():\n        if spec[\'src\'] == key:\n            gaussian_group_head(name, spec, *cache[spec[\'part\']])\n    del cache\n    CACHED_PARTS.pop(key)\n    gc.collect()\n\nfor key, dir_name, parts, C in ENC_HEADS:\n    print(f\'[{key}] {dir_name} parts {parts}\', flush=True)\n    cache = extract_parts(key, parts, encoder_loader(dir_name))\n    plain_and_twin(key, *concat_parts(cache, parts), C)\n    for name, spec in GAUSS_HEADS.items():\n        if spec[\'src\'] == key:\n            gaussian_group_head(name, spec, *cache[spec[\'part\']])\n    del cache\n    gc.collect()\n\n# ============================================================ 5. LoRA classifiers (inference only)\n@torch.no_grad()\ndef score_lora_checkpoints(trunk_spec, pool_layers, checkpoints):\n    """One truncated trunk, adapters swapped in turn. Plain readout -> blend member; centred readout -> twin on grouped rows;\n    within-Claude readout -> Claude gate."""\n    import importlib.util as iu\n    _find = iu.find_spec   # Kaggle\'s torchao build is rejected by peft; hide it (unused here)\n    iu.find_spec = lambda name, *a, **k: None if name.split(\'.\')[0] == \'torchao\' else _find(name, *a, **k)\n    from peft import PeftModel\n    import peft.import_utils as pi\n    pi.is_torchao_available = lambda *a, **k: False\n    from local_safetensors import load_file\n    model_type, H, keep = trunk_spec\n    base = find_checkpoint(model_type, H)\n    assert base, f\'no {model_type} base checkpoint under /kaggle/input\'\n    cfg = AutoConfig.from_pretrained(base)\n    cfg.num_hidden_layers, cfg.use_cache = keep, False\n    if getattr(cfg, \'layer_types\', None) is not None:\n        cfg.layer_types = cfg.layer_types[:keep]\n    trunk = AutoModel.from_pretrained(base, config=cfg, dtype=torch.float16, device_map=\'auto\').eval()\n    for key, marker, norms_file, anchor_file in checkpoints:\n        hits = [os.path.join(os.environ[\'NTL_LEGACY_QWEN24\'],\'COMPLETE.json\')] if key == \'lora24_public\' else sorted(glob.glob(\'/kaggle/input/\' + marker, recursive=True))\n        assert hits, f\'checkpoint for {key} not attached ({marker})\'\n        ck = pathlib.Path(os.path.dirname(hits[0]))\n        if str(ck).startswith(\'/kaggle/input/\'):\n            ck=stage_checkpoint(ck)\n\n        print(f\'[{key}] {ck}\', flush=True)\n        tok = AutoTokenizer.from_pretrained(ck / \'tokenizer\')\n        tok.padding_side = \'right\'\n        if tok.pad_token is None:\n            tok.pad_token = tok.eos_token\n        lm = PeftModel.from_pretrained(trunk, ck / \'adapter\').eval()\n        state = load_file(str(ck / norms_file))\n        norms = torch.nn.ModuleList([torch.nn.LayerNorm(H) for _ in pool_layers])\n        norms.load_state_dict({k[len(\'norms.\'):]: v for k, v in state.items() if k.startswith(\'norms.\')})\n        ro = np.load(ck / \'readouts.npz\')\n        assert list(ro[\'classes\']) == LABELS and list(ro[\'pool_layers\']) == pool_layers\n        dev = lm.get_input_embeddings().weight.device\n        norms = norms.to(dev).float()\n\n        def token_ids(texts):\n            return [ids + [tok.eos_token_id] for ids in tok(list(texts), add_special_tokens=False, truncation=True, max_length=LORA_MAX_LEN)[\'input_ids\']]\n\n        def features(texts, bs):\n            ids = token_ids(texts)\n            Z = np.zeros((len(ids), H * len(pool_layers)), dtype=np.float32)\n            order = np.argsort([len(t) for t in ids], kind=\'stable\')\n            for b0 in range(0, len(order), bs):\n                idx = order[b0:b0 + bs]\n                batch = tok.pad({\'input_ids\': [ids[i] for i in idx]}, return_tensors=\'pt\').to(dev)\n                with torch.autocast(\'cuda\', dtype=torch.float16):\n                    hs = lm(**batch, output_hidden_states=True, use_cache=False).hidden_states\n                m = batch[\'attention_mask\'].unsqueeze(-1).float()\n                m[:, 0] = 0   # the first token is masked out of the mean, as in training\n                Z[idx] = torch.cat([nm((hs[L].float() * m.to(hs[L].device)).sum(1).to(dev) / m.sum(1).to(dev))\n                                    for L, nm in zip(pool_layers, norms)], -1).cpu().numpy()\n                del hs, batch\n            return Z\n\n        if (ck / anchor_file).exists():   # the checkpoint\'s own features for training rows 0..2: our forward pass must reproduce them\n            saved = np.load(ck / anchor_file)[\'features\'].astype(np.float32)\n            ours = features(TR[:3], 3)\n            rel = float(np.abs(ours - saved).mean() / (np.abs(saved).mean() + 1e-9))\n            print(f\'  parity with the checkpoint anchors: max abs {float(np.abs(ours - saved).max()):.4f}, mean rel {rel:.5f}\', flush=True)\n            assert rel < 0.02, f\'{key}: our features do not match the checkpoint\'\n        Z = features(TE, LORA_BS)\n\n        if key == \'lora24_public\' and os.environ.get(\'NTL_NIGHT24_BANK_PATH\'):   # optional private readout bank\n            import hashlib,joblib\n            from readouts import predict_bank\n            bank=joblib.load(os.environ[\'NTL_NIGHT24_BANK_PATH\'])\n            with (ck/\'adapter/adapter_model.safetensors\').open(\'rb\') as fh:\n                assert hashlib.file_digest(fh,\'sha256\').hexdigest()==bank[\'adapter_sha256\']\n            assert bank[\'labels\']==LABELS and bank[\'layers\']==pool_layers\n            hp,hm=predict_bank(bank,Z,TE)\n            np.savez_compressed(\'/kaggle/working/night24_helper_probabilities.npz\',ids=test.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),probabilities=hp,**hm)\n            print(\'NIGHT24_READOUTS_REAL_TEST_COMPLETE\',len(Z),bank[\'synthetic_rows\'],flush=True)\n        add_member(key, softmax_rows(Z @ ro[\'plain_coef\'].T + ro[\'plain_intercept\']))\n        if \'claude_coef\' in ro.files:\n            lc = Z @ ro[\'claude_coef\'].T + ro[\'claude_intercept\']\n            GATES[\'claude_lora\'] = lc - np.log(np.exp(lc - lc.max(1, keepdims=True)).sum(1, keepdims=True)) - lc.max(1, keepdims=True)\n        if key in LORA_TWIN_W and GROUP_ROWS.any():\n            Zc = group_centre_dense(Z[GROUP_ROWS], G_TE[GROUP_ROWS])\n            add_member(key + \'_twin\', softmax_rows(Zc @ ro[\'center_coef\'].T + ro[\'center_intercept\']), rows=GROUP_ROWS, weight=LORA_TWIN_W[key])\n        trunk = lm.unload()\n        del lm, Z\n        gc.collect()\n        torch.cuda.empty_cache()\n    del trunk\n    gc.collect()\n    torch.cuda.empty_cache()\n\n\nif not SMOKE:\n    for trunk_spec, pool_layers, checkpoints in LORA_RUNS:\n        score_lora_checkpoints(trunk_spec, pool_layers, checkpoints)\n\n# ============================================================ 6. token-attention heads + UltraBERT\nclass TokenAttention(torch.nn.Module if torch is not None else object):   # the module of the public training kernels (lucasboesen/name-that-llm-train-tokattn, -tokattn2)\n    def __init__(self, k, h=4096):\n        super().__init__()\n        self.drop, self.proj, self.score, self.out = torch.nn.Dropout(0.2), torch.nn.Linear(h, 64), torch.nn.Linear(64, 1), torch.nn.Linear(128, k)\n\n    def forward(self, x, m):\n        h = torch.tanh(self.proj(self.drop(x)))\n        a = torch.softmax(self.score(h).squeeze(-1).masked_fill(~m, -1e4), -1).unsqueeze(-1)\n        mf = m.unsqueeze(-1).float()\n        return self.out(self.drop(torch.cat([(h * mf).sum(1) / mf.sum(1).clamp(min=1), (a * h).sum(1)], -1)))\n\n\n@torch.no_grad()\ndef token_attention_probs(texts, head_file, backbone, bs=8):\n    hits = sorted(glob.glob(\'/kaggle/input/**/\' + head_file, recursive=True))\n    assert hits, f\'training-kernel output holding {head_file} not attached\'\n    d = torch.load(hits[0], map_location=\'cpu\')\n    assert list(d[\'classes\']) == LABELS\n    decoder = d.get(\'decoder\', True)\n    path = find_checkpoint(\'qwen3_5\') if backbone == \'qwen3_5\' else find_checkpoint(None, dir_name=\'deberta-v3-large\')\n    assert path, f\'backbone {backbone} not attached\'\n    cfg = AutoConfig.from_pretrained(path)\n    if decoder:\n        tc = getattr(cfg, \'text_config\', None) or cfg\n        tc.num_hidden_layers = d[\'keep\']\n        if getattr(tc, \'layer_types\', None):\n            tc.layer_types = tc.layer_types[:d[\'keep\']]\n        lm = AutoModel.from_pretrained(path, config=cfg, dtype=torch.float16, device_map=\'auto\').eval()\n    else:\n        lm = AutoModel.from_pretrained(path, dtype=torch.float32).to(DEV).eval()\n    tok = AutoTokenizer.from_pretrained(path)\n    tok.padding_side = \'right\'\n    if tok.pad_token is None:\n        tok.pad_token = tok.eos_token\n    H = d.get(\'hidden\', 4096)\n    net = TokenAttention(K, H)\n    net.load_state_dict(d[\'state_dict\'])\n    dev = lm.get_input_embeddings().weight.device\n    net, mu, sd = net.to(dev).eval(), d[\'mu\'].float().to(dev), d[\'sd\'].float().to(dev)\n    ids = tok(list(texts), add_special_tokens=not decoder, truncation=True, max_length=d[\'max_len\'])[\'input_ids\']\n    P = np.zeros((len(texts), K))\n    order = np.argsort([len(t) for t in ids])\n    for b0 in range(0, len(order), bs):\n        idx = order[b0:b0 + bs]\n        batch = tok.pad({\'input_ids\': [ids[i] for i in idx]}, return_tensors=\'pt\').to(dev)\n        out = lm(input_ids=batch[\'input_ids\'], attention_mask=batch[\'attention_mask\'], output_hidden_states=True)\n        hs = (out.hidden_states if getattr(out, \'hidden_states\', None) is not None else out.language_model_outputs.hidden_states)[d[\'layer\']]\n        n = batch[\'attention_mask\'].sum(1)\n        T = max(int(n.max()) - 1, 1)\n        x = torch.zeros(len(idx), T, H, device=dev)\n        m = torch.zeros(len(idx), T, dtype=torch.bool, device=dev)\n        for r in range(len(idx)):   # first token dropped, states rounded through fp16, as in training\n            k = int(n[r]) - 1\n            if k > 0:\n                x[r, :k] = ((torch.nan_to_num(hs[r, 1:k + 1].to(dev).float()).half().float() - mu) / sd).clamp(-8, 8)\n                m[r, :k] = True\n        P[idx] = torch.softmax(net(x, m), -1).cpu().numpy()\n        del hs, out, batch\n    del lm\n    gc.collect()\n    torch.cuda.empty_cache()\n    return P\n\n\n@torch.no_grad()\ndef ultrabert_probs(texts):\n    from transformers import AutoModelForSequenceClassification\n    folds = None\n    for root in glob.glob(\'/kaggle/input/**/ultrabert-380m-ntl\', recursive=True):\n        fs = [f for f in sorted(glob.glob(os.path.join(root, \'fold*\'))) if os.path.exists(os.path.join(f, \'config.json\'))\n              and json.load(open(os.path.join(f, \'config.json\'))).get(\'model_type\') == \'ultrabert\']\n        if len(fs) == 5:\n            folds = fs\n            break\n    assert folds, \'ultrabert-380m-ntl (5 folds) not attached\'\n    tok = AutoTokenizer.from_pretrained(folds[0], trust_remote_code=True)\n    ids = tok(texts, truncation=True, max_length=512)[\'input_ids\']\n    order = np.argsort([len(e) for e in ids])\n    P = np.zeros((len(texts), K))\n    for fp in folds:\n        fp=str(stage_checkpoint(fp))\n        em = AutoModelForSequenceClassification.from_pretrained(fp, trust_remote_code=True, dtype=torch.float16, attn_implementation=\'sdpa\').to(DEV).eval()\n        cols = [LAB2I[em.config.id2label[i]] for i in range(len(em.config.id2label))]\n        for b0 in range(0, len(order), 32):\n            idx = order[b0:b0 + 32]\n            b = tok.pad({\'input_ids\': [ids[j] for j in idx]}, return_tensors=\'pt\').to(DEV)\n            P[np.ix_(idx, cols)] += em(**b).logits.float().softmax(-1).cpu().numpy() / len(folds)\n        del em\n        gc.collect()\n        torch.cuda.empty_cache()\n    return P\n\n\nif not SMOKE:\n    for key, head_file, backbone in TOKATTN_HEADS:\n        if TINY:\n            # One backbone load for the same fitting-only diagnosis and visible test.\n            combined=token_attention_probs(TR[:40]+TE,head_file,backbone)\n            acc=float(np.mean(combined[:40].argmax(1)==y[:40]))\n            print(f\'  {key} fitting-only diagnostic on40 rows: acc {acc:.2f}\',flush=True)\n            assert acc>.5, f\'{key} does not reproduce its training fit\'\n            prob=combined[40:]\n        else:\n            prob=token_attention_probs(TE,head_file,backbone)\n        add_member(key,prob)\n    np.save(\'/kaggle/working/comp_without_ultrabert.npy\',LOGIT)\n    json.dump({\'labels\':LABELS,\'ids\':test.id.astype(str).tolist(),\'seconds\':time.time()-T0,\'ultrabert_omitted\':True},open(\'/kaggle/working/comp_without_ultrabert_meta.json\',\'w\'))\n    add_member(\'ultrabert\', ultrabert_probs(TE))\n\n\n\nnp.savez_compressed(\'/kaggle/working/comp_members.npz\',ids=test.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),**MEMBER_EVIDENCE)\nnp.save(\'/kaggle/working/comp_logit.npy\', LOGIT)\nnp.savez_compressed(\'/kaggle/working/comp_gates.npz\', claude_labels=np.asarray([LABELS[i] for i in FAMILY_IDX[\'claude\']], dtype=str),\n                    **{k: v for k, v in GATES.items() if k in (\'claude\', \'claude_lora\')})   # for the optional Claude-gate experiment\njson.dump({\'labels\': LABELS, \'ids\': test[\'id\'].astype(str).tolist(), \'seconds\': time.time() - T0}, open(\'/kaggle/working/comp_meta.json\', \'w\'))\nprint(f\'COMP STACK DONE {time.time() - T0:.0f}s\', flush=True)', encoding="utf-8")
print("prepared",'/kaggle/working/comp_stack.py')

In [ ]:
if not PUBLIC_PREVIEW:

    # LoRA24 public adapter: rebuilt from the verified flat archive (sha256 per file)
    _manifests = list(Path('/kaggle/input').rglob('legacy_recovery_manifest.json'))
    assert len(_manifests) == 1, 'Attach one verified public legacy archive'
    _manifest = json.loads(_manifests[0].read_text())
    assert _manifest['independently_verified_published_migration_manifest']
    _legacy = Path('/kaggle/working/recovered_legacy/lora24/final')
    for _relative, _record in _manifest['files'].items():
        _rel = Path(_relative); assert not _rel.is_absolute() and '..' not in _rel.parts
        _origin = _manifests[0].parent / _record['flat_file']
        with _origin.open('rb') as _handle:
            assert hashlib.file_digest(_handle, 'sha256').hexdigest() == _record['sha256'], _relative
        _destination = _legacy / _rel; _destination.parent.mkdir(parents=True, exist_ok=True)
        shutil.copyfile(_origin, _destination)
    os.environ['NTL_LEGACY_QWEN24'] = str(_legacy)

    # Night24 readout bank (private upstream dataset): optional
    _bank_matches = list(Path('/kaggle/input').rglob('synthetic4404.joblib'))
    if len(_bank_matches) == 1:
        _bank_source = _bank_matches[0]
        _bank_audit = json.loads((_bank_source.parent / 'COMPLETE.json').read_text())
        assert _bank_audit['full_fit'] and not _bank_audit['fold_adapter'] and not _bank_audit['adapter_training']
        _record = _bank_audit['banks']['synthetic4404']
        with _bank_source.open('rb') as _f:
            assert hashlib.file_digest(_f, 'sha256').hexdigest() == _record['readout_sha256']
        _bank_local = Path('/kaggle/working/night24_bank'); _bank_local.mkdir(exist_ok=True)
        shutil.copyfile(_bank_source, _bank_local / _bank_source.name)
        shutil.copyfile(_bank_source.parent / 'readouts.py', Path('/kaggle/working/readouts.py'))
        os.environ['NTL_NIGHT24_BANK_PATH'] = str(_bank_local / _bank_source.name)
        STATUS['night24'] = 'attached'
    else:
        os.environ.pop('NTL_NIGHT24_BANK_PATH', None)
        STATUS['night24'] = 'skipped: private bank not attached'
        print('Night24 readout bank not attached: helper will be skipped', flush=True)

    _t = time.monotonic()
    _rc = subprocess.run([sys.executable, '-u', 'comp_stack.py'], cwd='/kaggle/working').returncode
    STATUS['comp_stack'] = 'ok' if _rc == 0 else f'skipped: exit code {_rc}'
    print('COMP STACK', STATUS['comp_stack'], f'{time.monotonic() - _t:.0f}s', flush=True)
    if _rc != 0 and STRICT:
        raise RuntimeError('comp stack failed')

In [ ]:
if not PUBLIC_PREVIEW:
    from stage_local_checkpoint import stage_checkpoint
    MEMBER_EVIDENCE = {}
    TOKEN_AUDIT = []

    import gc, glob, json, os, re, shutil, time

    # The scoring run has no internet, so transformers must never try to reach the Hub.
    os.environ["HF_HUB_OFFLINE"] = "1"
    os.environ["TRANSFORMERS_OFFLINE"] = "1"

    import numpy as np
    import pandas as pd
    import torch
    from torch import nn

    # Keep fp32 sums in fp16 matmuls. A T4 is otherwise allowed to sum in fp16, which drops digits in a deep model.
    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
    torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False

    T0 = time.time()
    INPUT_DIR = "/kaggle/input"

    # Blend weights. Every member adds weight * log(probability) to the score.
    W_SVC = 1.0
    LORAS = {  # checkpoint folder, then blend weight, centred twin weight, batch size and the base model to look for
        "lora_qwen3_14b":  dict(w=0.4, twin_w=0.1, bs=8, base=dict(model_type="qwen3", hidden_size=5120, num_hidden_layers=40)),
        "lora_qwen25_14b": dict(w=0.2, twin_w=0.1, bs=8, base=dict(model_type="qwen2", hidden_size=5120, num_hidden_layers=48)),
    }
    W_DEBERTA = 0.25
    TWIN = dict(thresh=0.85, min_size=2, max_size=18)   # topic clusters that get the centred twin

    # Column order of every checkpoint's head. Each checkpoint's meta.json is checked against it below.
    LABELS = ['gpt-5.5', 'gpt-5.6-luna', 'gpt-5.6-sol', 'gpt-5.6-terra', 'gpt-6-astra', 'gemini-3.1-pro-preview', 'gemini-3.6-flash', 'gemini-3.7-flash', 'gemini-3.8-flash', 'claude-opus-4-6', 'claude-opus-4-8', 'claude-opus-5', 'claude-sonnet-5', 'claude-fable-5', 'qwen3.8-max', 'glm-5.3', 'glm-5.3-flash', 'kimi-k3']
    K = len(LABELS)


    def find(pattern):
        if pattern in ('train.csv','test.csv'):
            for _root in ['/kaggle/input/competitions/name-that-llm','/kaggle/input/name-that-llm']:
                if Path(_root,pattern).exists():return str(Path(_root,pattern))
            raise FileNotFoundError('Official competition input '+pattern)
        hits = sorted(glob.glob(f"{INPUT_DIR}/**/{pattern}", recursive=True))
        if not hits:
            raise FileNotFoundError(pattern)
        return hits[0]


    def softmax(x):
        e = np.exp(x - x.max(1, keepdims=True))
        return e / e.sum(1, keepdims=True)


    def vote(name, P, w, rows=slice(None)):
        """Add one member's weighted log probabilities to the running score."""
        LOGIT[rows] += w * np.log(np.clip(P, 1e-9, 1))
        _e=np.zeros_like(LOGIT);_e[rows]=w*np.log(np.clip(P,1e-9,1));MEMBER_EVIDENCE[name]=_e
        np.savez_compressed('/kaggle/working/part_two_members.npz',ids=test.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),**MEMBER_EVIDENCE)
        print(f"{name}: weight {w}, done after {time.time() - T0:.0f}s", flush=True)


    train = pd.read_csv(find("train.csv"))
    test = pd.read_csv(find("test.csv"))
    test["text"] = test["text"].fillna("").astype(str)
    y = train["model"].map({l: i for i, l in enumerate(LABELS)}).values
    LOGIT = np.zeros((len(test), K))
    print("train", train.shape, "test", test.shape, "gpus", torch.cuda.device_count(), flush=True)

In [ ]:
if not PUBLIC_PREVIEW:
    from scipy.sparse import hstack
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.svm import LinearSVC

    SMALL_WORDS = set("a an the and or but if while although because as so than that which who whom whose this these those it its they them their theirs he him his she her hers we us our ours you your yours i me my mine in on at to from with without for of by over under between through into about is are was were be been being have has had do does did can could will would shall should may might must not no nor all any some each every both either neither other another such only also just very too more most less least here there when where why how what whether then now still yet even much many few little".split())
    VIEWS = [("char", 3, 5, 250000, 1.0), ("word", 1, 3, 180000, 0.7), ("shape", 2, 6, 60000, 0.5), ("small", 1, 3, 50000, 0.5)]


    def shape_text(t):
        """Every run of letters becomes X or x, so only the rhythm is left."""
        return re.sub(r"[^\W\d_]+", lambda m: "X" if m[0].isupper() else "x", t)


    def small_words_text(t):
        """Keep the small words and the punctuation, and replace every other word by CONTENT."""
        return " ".join(w.lower() if w.lower() in SMALL_WORDS or not w.isalnum() else "CONTENT"
                        for w in re.findall(r"\w+|[^\w\s]", t))


    pool = pd.read_parquet(find("pool_v4.parquet"))
    fit_texts = train.text.astype(str).tolist() + pool.text.tolist()
    fit_y = np.r_[y, pool.model.map({l: i for i, l in enumerate(LABELS)}).values]
    A, B = [], []
    for view, lo, hi, max_features, weight in VIEWS:
        prep = shape_text if view == "shape" else small_words_text if view == "small" else (lambda s: s)
        analyzer = "char" if view == "shape" else "word" if view == "small" else view
        opts = dict(analyzer=analyzer, ngram_range=(lo, hi), lowercase=False, sublinear_tf=True, min_df=2,
                    max_features=max_features, dtype=np.float32)
        if analyzer == "word":
            opts["token_pattern"] = r"(?u)\b\w+\b|[^\w\s]"
        vec = TfidfVectorizer(**opts)
        A.append(vec.fit_transform([prep(t) for t in fit_texts]) * weight)
        B.append(vec.transform([prep(t) for t in test.text]) * weight)
    svc = LinearSVC(C=0.5, random_state=42, max_iter=5000, class_weight="balanced").fit(hstack(A, format="csr"), fit_y)
    vote("svc", softmax(svc.decision_function(hstack(B, format="csr"))), W_SVC)
    del A, B, svc, pool
    gc.collect()

In [ ]:
if not PUBLIC_PREVIEW:
    from safetensors.torch import load_file
    from transformers import AutoConfig, AutoModel, AutoTokenizer


    def find_base(spec):
        """The attached base model whose config.json matches spec."""
        for cfg in sorted(glob.glob(f"{INPUT_DIR}/**/config.json", recursive=True)):
            try:
                c = json.load(open(cfg))
            except Exception:
                continue
            if all(c.get(k) == v for k, v in spec.items()):
                return os.path.dirname(cfg)
        raise FileNotFoundError(f"attach the base model {spec}")


    def load_trunk(base_dir, keep, dtype):
        """The first keep decoder layers, split over both T4s when they do not fit on one."""
        mc = AutoConfig.from_pretrained(base_dir)
        mc.num_hidden_layers, mc.use_cache = keep, False
        if getattr(mc, "layer_types", None) is not None:
            mc.layer_types = mc.layer_types[:keep]
        H, I, V = mc.hidden_size, mc.intermediate_size, mc.vocab_size
        hd = getattr(mc, "head_dim", H // mc.num_attention_heads)
        q, kv = mc.num_attention_heads * hd, mc.num_key_value_heads * hd
        gib = 2 * (V * H + keep * (2 * H * q + 2 * H * kv + 3 * H * I)) / 2 ** 30
        if gib > 13 and torch.cuda.device_count() > 1:
            mem = {i: "13GiB" for i in range(torch.cuda.device_count())}
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype, device_map="auto", max_memory=mem).eval()
        else:
            model = AutoModel.from_pretrained(base_dir, config=mc, dtype=dtype).to("cuda:0").eval()
        model.norm = nn.Identity()   # the checkpoints were trained without the final norm
        print(f"trunk of {keep} layers, {gib:.1f} GiB of weights", flush=True)
        return model


    @torch.no_grad()
    def merge_lora(model, adapter_dir):
        """W becomes W + B A alpha / r for every adapted linear layer."""
        acfg = json.load(open(os.path.join(adapter_dir, "adapter_config.json")))
        scale = acfg["lora_alpha"] / acfg["r"]
        sd = load_file(os.path.join(adapter_dir, "adapter_model.safetensors"))
        modules = dict(model.named_modules())
        for k, A in sd.items():
            if ".lora_A." not in k:
                continue
            name = k.split(".lora_A.")[0]
            lin = modules[name[name.index("layers."):]]
            Bm = sd[k.replace(".lora_A.", ".lora_B.")]
            delta = (Bm.float().to(lin.weight.device) @ A.float().to(lin.weight.device)) * scale
            lin.weight.add_(delta.to(lin.weight.dtype))


    @torch.no_grad()
    def lora_features(ckpt_dir, base_dir, texts, bs):
        """Pooled features Z, one row per passage, and the checkpoint's heads."""
        meta = json.load(open(os.path.join(ckpt_dir, "meta.json")))
        assert meta["labels"] == LABELS, "label order differs from the checkpoint's heads"
        cfg = meta["cfg"]
        dtype = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
        tok = AutoTokenizer.from_pretrained(base_dir)
        tok.padding_side = "right"
        if tok.pad_token is None:
            tok.pad_token = tok.eos_token
        model = load_trunk(base_dir, cfg["keep"], dtype)
        merge_lora(model, os.path.join(ckpt_dir, "adapter"))
        heads = load_file(os.path.join(ckpt_dir, "heads.safetensors"))
        layers, H = cfg["layers"], model.config.hidden_size
        norms = nn.ModuleList([nn.LayerNorm(H) for _ in layers]).to("cuda:0").float()
        norms.load_state_dict({k[len("norms."):]: v for k, v in heads.items() if k.startswith("norms.")})
        ids = tok(list(texts), add_special_tokens=False, truncation=True, max_length=cfg["max_len"])["input_ids"]
        # The end token from training, not this tokenizer's. Kaggle's Qwen3-14B-Base carries the chat model's end token.
        eos_id = cfg.get("eos_id", tok.eos_token_id)
        assert 'eos_id' in cfg, 'Training end-token metadata required'
        TOKEN_AUDIT.append(dict(checkpoint=ckpt_dir,training_eos_id=int(eos_id),base_tokenizer_eos_id=int(tok.eos_token_id)))
        Path('/kaggle/working/token_audit.json').write_text(json.dumps(TOKEN_AUDIT,indent=2))
        print(f"end token {eos_id}, this tokenizer's is {tok.eos_token_id}", flush=True)
        ids = [x + [eos_id] for x in ids]
        order = np.argsort([len(x) for x in ids], kind="stable")   # similar lengths together means less padding
        Z = np.zeros((len(ids), H * len(layers)), np.float32)

        def embed(idx):
            enc = tok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to(model.embed_tokens.weight.device)
            hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
            if not all(torch.isfinite(hs[l]).all() for l in layers):   # fp16 overflow, so redo this batch in bf16
                model.to(torch.bfloat16)
                hs = model(**enc, output_hidden_states=True, use_cache=False).hidden_states
                model.to(dtype)
            m = enc["attention_mask"].unsqueeze(-1).float().to("cuda:0")
            m[:, 0] = 0   # the first token soaks up attention whatever it is, so leave it out of the average
            den = m.sum(1).clamp(min=1)
            Z[idx] = torch.cat([nm((hs[l].to("cuda:0").float() * m).sum(1) / den) for l, nm in zip(layers, norms)], -1).cpu().numpy()

        for s in range(0, len(order), bs):
            idx = order[s:s + bs]
            try:
                embed(idx)
            except torch.cuda.OutOfMemoryError:   # the longest batch can overflow a 15 GB T4, so go one passage at a time
                torch.cuda.empty_cache()
                for i in idx:
                    embed([i])
        del model
        gc.collect()
        torch.cuda.empty_cache()
        return Z, {k: v.float().numpy() for k, v in heads.items() if not k.startswith("norms.")}

In [ ]:
if not PUBLIC_PREVIEW:
    from scipy.cluster.hierarchy import fcluster, linkage
    from scipy.spatial.distance import squareform

    if len(test) >= 2:
        Xt = TfidfVectorizer(stop_words="english", sublinear_tf=True).fit_transform(test.text.tolist())
        D = np.clip(1 - (Xt @ Xt.T).toarray(), 0, None)
        np.fill_diagonal(D, 0)
        groups = fcluster(linkage(squareform(D, checks=False), method="average"), TWIN["thresh"], criterion="distance")
        del D
    else:
        groups = np.ones(len(test), dtype=int)
    size = np.bincount(groups)[groups]
    twin_rows = (size >= TWIN["min_size"]) & (size <= TWIN["max_size"])
    print(f"topic clusters: {int((size == 1).sum())} passages alone, {int(twin_rows.sum())} in clusters of 2 to 18", flush=True)

    for ckpt_name, mc in LORAS.items():
        ckpt = os.path.dirname(find(f"{ckpt_name}/meta.json"))
        ckpt = str(stage_checkpoint(ckpt))
        Z, heads = lora_features(ckpt, find_base(mc["base"]), test.text.tolist(), mc["bs"])
        vote(ckpt_name, softmax(Z @ heads["plain.weight"].T + heads["plain.bias"]), mc["w"])
        if twin_rows.any():
            Zc = Z.copy()
            for g in np.unique(groups[twin_rows]):
                Zc[groups == g] -= Zc[groups == g].mean(0)
            centred = Zc[twin_rows] @ heads["centered.weight"].T + heads["centered.bias"]
            vote(ckpt_name + " twin", softmax(centred), mc["twin_w"], rows=twin_rows)
        del Z
        gc.collect()

In [ ]:
if not PUBLIC_PREVIEW:
    ck = os.path.dirname(find("deberta_v3_large/meta.json"))
    if ck.startswith("/kaggle/input"):   # one long copy to local disk beats many small reads from the dataset mount
        shutil.copytree(ck, "/tmp/deberta_v3_large", dirs_exist_ok=True)
        ck = "/tmp/deberta_v3_large"
    meta = json.load(open(os.path.join(ck, "meta.json")))
    assert meta["labels"] == LABELS, "label order differs from the checkpoint's head"
    etok = AutoTokenizer.from_pretrained(os.path.join(ck, "encoder"))
    enc = AutoModel.from_pretrained(os.path.join(ck, "encoder"), dtype=torch.float32).to("cuda:0").eval()
    head = load_file(os.path.join(ck, "head.safetensors"))
    W, b = head["weight"].float().numpy(), head["bias"].float().numpy()
    ids = etok(test.text.tolist(), truncation=True, max_length=meta["max_len"])["input_ids"]
    order = np.argsort([len(x) for x in ids], kind="stable")
    Z = np.zeros((len(ids), enc.config.hidden_size), np.float32)
    with torch.inference_mode():
        for s in range(0, len(order), 32):
            idx = order[s:s + 32]
            bt = etok.pad({"input_ids": [ids[i] for i in idx]}, return_tensors="pt").to("cuda:0")
            m = bt["attention_mask"].unsqueeze(-1).float()
            Z[idx] = ((enc(**bt).last_hidden_state * m).sum(1) / m.sum(1).clamp(min=1)).cpu().numpy()
    vote("deberta", softmax(Z @ W.T + b), W_DEBERTA)
    del enc
    gc.collect()
    torch.cuda.empty_cache()

In [ ]:
if not PUBLIC_PREVIEW:
    def sinkhorn(P, iters=300):
        """One scale per class, so that every class gets an equal share of the probability. Rows are renormalised."""
        c = np.ones(K)
        target = np.full(K, len(P) / K)
        for _ in range(iters):
            R = P * c
            R /= R.sum(1, keepdims=True)
            c *= target / R.sum(0)
        R = P * c
        return R / R.sum(1, keepdims=True)


    P = softmax(LOGIT)
    if len(test) >= 50:   # skip on the 3 row public test.csv
        P = sinkhorn(P)
    sub = pd.DataFrame({"id": test["id"].astype(str), "model": np.array(LABELS)[P.argmax(1)]})
    assert len(sub) == len(test) and sub["model"].isin(LABELS).all()
    sub.to_parquet("submission.parquet", index=False)
    print(sub["model"].value_counts().to_string())
    print(f"done in {time.time() - T0:.0f}s")
    assert set(MEMBER_EVIDENCE) >= {'svc','lora_qwen3_14b','lora_qwen25_14b','deberta'}
    assert np.isfinite(LOGIT).all() and np.isfinite(P).all() and np.allclose(P.sum(1),1)
    assert np.allclose(sum(MEMBER_EVIDENCE.values()),LOGIT,rtol=1e-7,atol=1e-7)
    np.savez_compressed('/kaggle/working/part_two_probabilities.npz',ids=test.id.to_numpy(dtype=str),labels=np.asarray(LABELS,dtype=str),logits=LOGIT,probabilities=P,groups=groups)
    _data=Path(find('test.csv')).parent
    _sha=lambda p:__import__('hashlib').sha256(Path(p).read_bytes()).hexdigest()
    _evidence=dict(rows=len(test),format_only=len(test)==3,accuracy_validation=False,train_sha256=_sha(_data/'train.csv'),test_sha256=_sha(_data/'test.csv'),pool_sha256=_sha(find('pool_v4.parquet')),members=list(MEMBER_EVIDENCE),training_token_audit=TOKEN_AUDIT,full_actual_model_inference=True)
    Path('/kaggle/working/part_two_verification.json').write_text(json.dumps(_evidence,indent=2))
    print('PART_TWO_ALL_REQUIRED_MEMBERS_VERIFIED',json.dumps(_evidence),flush=True)

In [ ]:
if not PUBLIC_PREVIEW:
    def atlas_scores(bundle, rows, budget):
        """Source-conditioned likelihood probabilities (public ATLAS bundle format) for test rows `rows`, in our label order.
        Rows are scored shortest-first; raises if the token-proportional projection would overrun `budget` (s since start)."""
        from stage_local_checkpoint import stage_checkpoint
        bundle = str(stage_checkpoint(bundle))
        sys.path.insert(0, bundle)
        for _m in [m_ for m_ in list(sys.modules) if m_.startswith(('ntl_atlas', 'ntl_rlcd'))]:
            del sys.modules[_m]
        from ntl_atlas.model import load_model as _atlas_load
        from ntl_rlcd.data import LABELS as _alab
        from safetensors.torch import load_file as _lf
        sys.path.remove(bundle)
        _cfg = json.load(open(os.path.join(bundle, 'atlas_bundle.json')))['training_config']
        _am, _atok = _atlas_load(a.atlas_base, _alab, _cfg, device='auto')
        try:
            _am.load_trainable(_lf(os.path.join(bundle, 'adapters.safetensors')))
            _am.eval()
            _enc = _atok(test.text.values[rows].tolist(), truncation=True, max_length=_cfg['max_length'])['input_ids']
            _ord = np.argsort([len(t) for t in _enc], kind='stable')
            out = np.zeros((len(rows), 18), np.float32)
            _t1 = time.time()
            _ctok = np.cumsum(np.array([len(_enc[int(i)]) for i in _ord], np.float64))
            for _b in range(0, len(_ord), a.atlas_bs):
                _idx = _ord[_b:_b + a.atlas_bs]
                _pk = _atok.pad({'input_ids': [_enc[int(i)] for i in _idx]}, padding=True, return_tensors='pt').to(_am.device)
                with torch.inference_mode():
                    _p, _ = _am.score(_pk['input_ids'], _pk['attention_mask'], source_chunk=a.atlas_chunk)
                out[_idx] = _p.float().cpu().numpy()
                _done = _b + len(_idx)
                if _b % (25 * a.atlas_bs) == 0:
                    _proj = time.time() - T0 + (time.time() - _t1) / _ctok[_done - 1] * (_ctok[-1] - _ctok[_done - 1]) * 1.1
                    log(f'atlas {os.path.basename(bundle)} {_done}/{len(_ord)} projected finish {_proj:.0f}s')
                    if _done > 50 and _proj > budget:
                        raise RuntimeError('atlas would overrun the time budget')
                    if a.atlas_probe and _done >= a.atlas_probe:
                        raise RuntimeError(f'atlas probe stop; projected {_proj:.0f}s')
            assert np.isfinite(out).all() and np.allclose(out.sum(1), 1, atol=0.0001)
            return out[:, [_alab.index(l) for l in LABELS]]
        finally:
            del _am
            gc.collect()
            torch.cuda.empty_cache()


    def _skip(name, why):
        STATUS[name] = f'skipped: {why}'
        print(f'WARNING {name} skipped: {why}', flush=True)
        if STRICT:
            raise RuntimeError(f'{name}: {why}')


    try:
        _main_raw = softmax(LOGIT)                    # raw main probabilities, before balancing
        P = _main_raw.copy()
        _comp = np.zeros_like(P)
        # ---- comp stack
        if STATUS.get('comp_stack') == 'ok':
            _cm = json.loads(Path('/kaggle/working/comp_meta.json').read_text())
            assert _cm['ids'] == test.id.astype(str).tolist() and set(_cm['labels']) == set(LABELS)
            _comp = softmax(np.load('/kaggle/working/comp_logit.npy', allow_pickle=False)[:, [_cm['labels'].index(l) for l in LABELS]])
            _members = np.load('/kaggle/working/comp_members.npz', allow_pickle=False)
            _required = {'stylegram', 'stylometry_lgb', 'debv3l', 'electl', 'desk', 'lora24_public', 'lora24_seed7', 'lora24_claude',
                         'lora20_q25_instyle', 'tokattn_q35', 'tokattn_deb', 'ultrabert'}
            assert _required.issubset(_members.files), sorted(_required - set(_members.files))
            P = MIX_MAIN * _main_raw + MIX_COMP * _comp
        else:
            _skip('comp_mix', 'comp stack ' + STATUS.get('comp_stack', 'not run').removeprefix('skipped: '))
        _before_atlas = P.copy()

        # ---- ATLAS on low-margin rows
        _margins = np.sort(P, axis=1); _margins = _margins[:, -1] - _margins[:, -2]
        _atlas_rows = np.flatnonzero(_margins < ATLAS_MARGIN) if len(test) >= 50 else np.arange(len(test))
        _atlas_prob = np.zeros_like(P)
        _atlas_bundles = list(Path('/kaggle/input').rglob('atlas_bundle.json'))
        _base_configs = [p for p in Path('/kaggle/input').rglob('config.json') if '/qwen-3/' in str(p) and '/8b-base/' in str(p)]
        _remaining = TIME_BUDGET_S - (time.monotonic() - OVERALL_START) - 600
        if len(_atlas_bundles) != 1 or len(_base_configs) != 1:
            _skip('atlas', f'{len(_atlas_bundles)} bundles / {len(_base_configs)} Qwen3-8B bases found')
        elif _remaining <= 0:
            _skip('atlas', 'no time left')
        elif len(_atlas_rows) and ATLAS_W > 0:
            from types import SimpleNamespace
            a = SimpleNamespace(atlas_base=str(_base_configs[0].parent), atlas_bs=4, atlas_chunk=3, atlas_probe=0)
            log = lambda *x: print(*x, flush=True)
            try:
                _atlas_prob[_atlas_rows] = atlas_scores(str(_atlas_bundles[0].parent), _atlas_rows, (time.time() - T0) + _remaining)
                P[_atlas_rows] = (1 - ATLAS_W) * P[_atlas_rows] + ATLAS_W * _atlas_prob[_atlas_rows]
                STATUS['atlas'] = f'ok on {len(_atlas_rows)} rows'
            except RuntimeError as e:          # time-budget guard inside atlas_scores
                _atlas_prob[:] = 0
                _skip('atlas', repr(e))
        assert np.isfinite(P).all() and np.allclose(P.sum(1), 1)

        # ---- Night24 helper
        _hp = np.zeros_like(P)
        _unchanged_top1 = P.copy()
        _hpath = Path('/kaggle/working/night24_helper_probabilities.npz')
        if HELPER_W > 0 and _hpath.exists():
            _helper = np.load(_hpath, allow_pickle=False)
            assert _helper['ids'].tolist() == test.id.astype(str).tolist() and set(_helper['labels']) == set(LABELS)
            _hp = _helper['probabilities'][:, [_helper['labels'].tolist().index(l) for l in LABELS]]
            assert _hp.shape == P.shape and np.isfinite(_hp).all() and np.allclose(_hp.sum(1), 1)
            P = (1 - HELPER_W) * P + HELPER_W * _hp
            STATUS['night24_helper'] = 'ok'
        else:
            STATUS['night24_helper'] = 'skipped: ' + ('weight 0' if HELPER_W == 0 else 'bank not attached')
            print('Night24 helper', STATUS['night24_helper'], flush=True)

        # ---- balance and decide
        _before_balancing = P.copy()
        if len(test) >= 50:
            P = sinkhorn(P, SINKHORN_ITERS)
        pred = P.argmax(1)

        # ---- EXPERIMENT (off by default): within-Claude re-pick on Claude-predicted rows
        _gpath = Path('/kaggle/working/comp_gates.npz')
        if CLAUDE_GATE and _gpath.exists():
            _g = np.load(_gpath, allow_pickle=False)
            if 'claude' in _g.files:
                _cidx = np.array([LABELS.index(l) for l in _g['claude_labels']])
                _s = np.log(np.clip(P[:, _cidx], 1e-12, 1)) + CLAUDE_GATE_LAM['style'] * _g['claude']
                if 'claude_lora' in _g.files:
                    _s = _s + CLAUDE_GATE_LAM['lora'] * _g['claude_lora']
                _isc = np.isin(pred, _cidx); _new = _cidx[_s.argmax(1)]
                print('claude gate changed', int((_new[_isc] != pred[_isc]).sum()), 'of', int(_isc.sum()), 'Claude predictions', flush=True)
                pred[_isc] = _new[_isc]
                STATUS['claude_gate'] = 'ok'

        # ---- write
        Path('/kaggle/working/submission.parquet').replace('/kaggle/working/part_two_anchor_submission.parquet')
        sub = pd.DataFrame(dict(id=test.id.astype(str), model=np.array(LABELS)[pred]))
        assert sub.id.is_unique and sub.model.isin(train.model.unique()).all() and len(sub) == len(test)
        sub.to_parquet('/kaggle/working/submission.parquet', index=False)
        np.savez_compressed('/kaggle/working/enhanced_probabilities.npz', ids=test.id.to_numpy(dtype=str), labels=np.asarray(LABELS),
                            main_raw=_main_raw, complement=_comp, before_atlas=_before_atlas, atlas=_atlas_prob, atlas_rows=_atlas_rows,
                            helper=_hp, before_balancing=_before_balancing, probabilities=P)
        changed = int((pred != np.load('/kaggle/working/part_two_probabilities.npz')['probabilities'].argmax(1)).sum())
        print(f'final submission written; {changed} of {len(sub)} rows differ from the part-two anchor', flush=True)
    except Exception as e:
        if STRICT:
            raise
        STATUS['fusion'] = f'FAILED, anchor submission kept: {e!r}'
        print('WARNING', STATUS['fusion'], flush=True)
        if not Path('/kaggle/working/submission.parquet').exists():
            Path('/kaggle/working/part_two_anchor_submission.parquet').replace('/kaggle/working/submission.parquet')

    Path('/kaggle/working/enhanced_verification.json').write_text(json.dumps(dict(
        rows=len(test), status=STATUS, mix=dict(main=MIX_MAIN, comp=MIX_COMP, atlas=ATLAS_W, atlas_margin=ATLAS_MARGIN, helper=HELPER_W),
        claude_gate=CLAUDE_GATE, elapsed_seconds=time.monotonic() - OVERALL_START), indent=2))
    print(json.dumps(STATUS, indent=2))

In [ ]:
if not PUBLIC_PREVIEW:
    import os, gc, json, time
    from pathlib import Path
    import numpy as np
    import pandas as pd
    from scipy.special import softmax

    WORK23 = Path('/kaggle/working')
    report23 = {'version':'v23-ultrabert-consensus','status':'reference','additional_model':'pjmathematician/ultrabert-380m-ntl'}
    ref23 = pd.read_parquet(WORK23/'submission.parquet')
    expected23 = test.id.astype(str).to_numpy()
    labels23 = np.asarray(LABELS, dtype=str)
    assert ref23.id.astype(str).tolist() == expected23.tolist()
    assert ref23.id.is_unique and ref23.model.isin(labels23).all()
    ref23.to_parquet(WORK23/'candidate_reference_0639.parquet',index=False)

    def safe23(x):
        x=np.asarray(x,dtype=np.float64)
        if x.shape != (len(test),len(labels23)) or not np.isfinite(x).all() or (x<0).any():
            raise ValueError('probability shape, range, or finite check failed')
        x=np.clip(x,1e-12,None)
        return x / x.sum(1,keepdims=True)

    def paths23():
        roots=[]
        for pattern in ('/kaggle/input/*ultrabert*','/kaggle/input/*ultra*bert*','/kaggle/input/*380m*ntl*'):
            roots.extend(Path('/').glob(pattern.lstrip('/')))
        for root in dict.fromkeys(roots):
            fs=[]
            for k in range(5):
                matches=[d for d in (root/f'fold{k}',root/'models'/f'fold{k}',root/'checkpoints'/f'fold{k}') if (d/'config.json').exists()]
                if not matches:
                    matches=list(root.glob(f'**/fold{k}/config.json'))[:2]
                    matches=[m.parent for m in matches]
                if matches:
                    fs.append(matches[0])
            if len(fs)==5:
                return root,fs
        return None,[]

    def class_map23(folder):
        cfg=json.loads((folder/'config.json').read_text())
        id2label=cfg.get('id2label',{})
        sequence=[str(id2label.get(str(i),'')).strip() for i in range(len(labels23))]
        if set(sequence)==set(labels23) and len(set(sequence))==len(labels23):
            return [sequence.index(label) for label in labels23]
        for p in [folder/'labels.json',folder/'label_classes.json',folder.parent/'labels.json',folder.parent/'label_classes.json']:
            if p.exists():
                val=json.loads(p.read_text())
                if isinstance(val,dict):
                    val=val.get('classes',val.get('labels',[]))
                if isinstance(val,list) and set(map(str,val))==set(labels23):
                    val=list(map(str,val))
                    return [val.index(label) for label in labels23]
        return None

    def infer23(folder, order):
        import torch
        from transformers import AutoTokenizer, AutoModelForSequenceClassification
        tokenizer=AutoTokenizer.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False)
        model=AutoModelForSequenceClassification.from_pretrained(str(folder),local_files_only=True,trust_remote_code=False,torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32)
        model.eval()
        device='cuda:0' if torch.cuda.is_available() else 'cpu'
        model.to(device)
        if order is None:
            from scipy.optimize import linear_sum_assignment
            probe=train.groupby('model', group_keys=False).sample(n=4, random_state=23)
            probe_text=probe.text.fillna('').astype(str).tolist()
            probe_truth=probe.model.astype(str).to_numpy()
            raw_probe=[]
            with torch.inference_mode():
                for j in range(0,len(probe_text),24):
                    bt=tokenizer(probe_text[j:j+24],padding=True,truncation=True,max_length=384,return_tensors='pt')
                    bt={k:v.to(device) for k,v in bt.items()}
                    raw_probe.append(torch.softmax(model(**bt).logits.float(),dim=-1).cpu().numpy())
            raw_probe=np.concatenate(raw_probe)
            similarity=np.stack([raw_probe[probe_truth==name].mean(0) for name in labels23])
            row_ids,col_ids=linear_sum_assignment(-similarity)
            if len(col_ids)!=len(labels23) or np.any(similarity[row_ids,col_ids]<.12):
                raise RuntimeError('unable to recover UltraBERT output label mapping safely')
            order=np.empty(len(labels23),dtype=int)
            order[row_ids]=col_ids
        txt=test.text.fillna('').astype(str).tolist()
        probs=[]
        with torch.inference_mode():
            for i in range(0,len(txt),24):
                toks=tokenizer(txt[i:i+24],padding=True,truncation=True,max_length=384,return_tensors='pt')
                toks={k:v.to(device) for k,v in toks.items()}
                out=model(**toks).logits.float()
                probs.append(torch.softmax(out,dim=-1)[:,order].cpu().numpy())
        del model,tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        return safe23(np.concatenate(probs,axis=0))

    def rerank23(reference, new, agreement, weight):
        base=safe23(reference)
        supplement=safe23(new)
        top2=np.argsort(base,axis=1)[:,-2:]
        top=top2[:,1]
        alternative=top2[:,0]
        margin=base[np.arange(len(base)),top]-base[np.arange(len(base)),alternative]
        supplement_best=supplement.argmax(1)
        hard=(supplement_best==alternative)&(agreement>=4)&(margin<.10)
        score=np.log(np.clip(base,1e-12,1))
        u=np.log(np.clip(supplement,1e-12,1))
        rows=np.where(hard)[0]
        if len(rows):
            rivals=alternative[rows]
            champions=top[rows]
            signal=(u[rows,rivals]-u[rows,champions]).clip(0,3)
            score[rows,rivals]+=weight*signal
        return safe23(softmax(score,axis=1)),int(hard.sum())

    try:
        if len(test)<50:
            raise RuntimeError('Kaggle preview: preserve original output')
        root,folds=paths23()
        if not folds:
            raise RuntimeError('optional public UltraBERT input is not attached')
        remaining=TIME_BUDGET_S-(time.monotonic()-OVERALL_START)
        report23['remaining_seconds']=int(remaining)
        if remaining<1500:
            raise RuntimeError('insufficient time for optional five-fold inference')
        with np.load(WORK23/'enhanced_probabilities.npz',allow_pickle=False) as cache:
            baseprob=safe23(cache['probabilities'])
            if 'ids' in cache and cache['ids'].astype(str).tolist()!=expected23.tolist():
                raise ValueError('cache IDs mismatch')
            if 'labels' in cache and list(cache['labels'].astype(str))!=list(labels23):
                raise ValueError('cache label order mismatch')
        if not np.array_equal(labels23[baseprob.argmax(1)],ref23.model.to_numpy()):
            raise ValueError('reference notebook probabilities do not reproduce the original predictions')
        mapping=[class_map23(f) for f in folds]
        foldprobs=[]
        for i,(directory,idx) in enumerate(zip(folds,mapping)):
            if TIME_BUDGET_S-(time.monotonic()-OVERALL_START)<240:
                raise RuntimeError('remaining time too small; preserve baseline')
            foldprobs.append(infer23(directory,idx))
            print('UltraBERT fold',i,'complete',flush=True)
        folds_array=np.stack(foldprobs,axis=0)
        uber=safe23(folds_array.mean(0))
        agreement=(folds_array.argmax(2)==uber.argmax(1)[None,:]).sum(0)
        report23['folds']=len(foldprobs)
        proposals={}
        for power in (.10,.18,.28):
            proba,eligible=rerank23(baseprob,uber,agreement,power)
            name=f'ultra_{int(power*100):02d}'
            out=pd.DataFrame({'id':expected23,'model':labels23[proba.argmax(1)]})
            out.to_parquet(WORK23/f'candidate_{name}.parquet',index=False)
            proposals[name]=out
            report23[name]={'eligible':eligible,'changes':int((out.model.to_numpy()!=ref23.model.to_numpy()).sum())}
        choice='ultra_18'
        proposals[choice].to_parquet(WORK23/'submission.parquet',index=False)
        np.savez_compressed(WORK23/'v23_ultra_probs.npz',ids=expected23,labels=labels23,ultrabert=uber,reference=baseprob,agreement=agreement)
        report23['status']='ok'
        report23['chosen']=choice
    except Exception as exc:
        report23['status']='original_0639_preserved'
        report23['reason']=str(exc)
        ref23.to_parquet(WORK23/'submission.parquet',index=False)
    final23=pd.read_parquet(WORK23/'submission.parquet')
    assert final23.id.astype(str).tolist()==expected23.tolist() and final23.id.is_unique and final23.model.isin(labels23).all()
    (WORK23/'v23_diagnostics.json').write_text(json.dumps(report23,indent=2,ensure_ascii=False))
    print('v23',json.dumps(report23,ensure_ascii=False),flush=True)
